# ISIC-2017 · M-TACS full_both · export từ best.pt đã train

Attach **output cũ thành Kaggle Dataset**, rồi chạy các cell theo thứ tự. Notebook **không train lại**,
không tạo lại CSV, không thay text embedding và không sửa output cũ. Khôi phục đúng ViT-L/16, 448×448, K=1;
query text + candidate text vào fusion và selector như checkpoint đã lưu. Epoch, metric, split và pool đọc từ run thật,
không lấy số liệu của TACS. Run bị dừng vẫn ghi `training_complete=false`; `ready` chỉ xác nhận export đã kiểm nạp lại.

**Inputs cần attach:**
- Output M-TACS trong ảnh: `best.pt`, `protocol.json`, `run_binding.json`, `candidate_pool.csv`, ba split CSV,
  `embedded_runtime/`, `history.csv` và `shared/caption_generation/` (giữ CSV, text_cache, settings, status, generation_protocol).
  `last.pt` không bắt buộc; có thì chỉ đọc metadata. Có `shared/candidate_bank.pt` sẽ dùng lại, thiếu thì encode đúng pool cũ.
- ISIC-2017 ảnh + mask gốc; DINOv3 source và frozen `dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth` đúng run.
- **Đúng Florence-2-base Microsoft + CLIP ViT-B/32 đã dùng lúc tạo caption**, cùng offline wheels từ notebook cũ
  (`transformers 4.49.x`, `tokenizers 0.21.x`, `huggingface-hub<1`, và các dependency cần thiết).
  Output train thường không chứa lại hai model pretrained này: attach thêm Input gốc.

Output đủ cho web: backbone, selector, segmentor (mọi context block/DPT/text projector/gate), candidate bank và ảnh,
mask GT để đối chiếu, captions.csv, text cache, Florence/CLIP + processor/tokenizer, runtime và hai map attention thật.
Attention không cần model train riêng. Caption là mô tả tự động từ pixels, không phải nhãn GT segmentation.
ZIP được ghi thành từng phần 256 MiB để tải riêng, không phải tải toàn bộ folder lớn.


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["PIP_NO_INDEX"] = "1"
import sys, importlib.util, json
from pathlib import Path
missing = [name for name in ("torch", "numpy", "PIL", "tqdm") if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError("Missing offline packages: " + ", ".join(missing))
import torch
if not hasattr(torch.nn.functional, "scaled_dot_product_attention"):
    raise RuntimeError("PyTorch with SDPA is required")
torch.set_num_threads(2)
torch.set_float32_matmul_precision("highest")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available())


## 1. Đường dẫn

`SOURCE_RUN_DIR=""` tự tìm đúng một run **mtacs** trong `/kaggle/input`; nếu có nhiều run, điền folder Input bạn vừa upload
hoặc folder cụ thể chứa `best.pt`. Ví dụ `/kaggle/input/datasets/TEN_CUA_BAN/OUTPUT_MTACS/outputs/isic2017_mtacs_...`.
Không điền đường dẫn tới source DINOv3 vào biến này. Giữ OUTPUT_DIR riêng với Input cũ.

Dataset/model có thể tự dò tìm; nếu có nhiều bản, điền đường dẫn chính xác. Hash phải khớp run đã train.
Không cần hạ Transformers trong kernel: chỉ caption subprocess dùng 4.49 từ wheels offline.


In [ ]:
SOURCE_RUN_DIR = os.getenv("ISIC_MTACS_SAVED_RUN", "")
CAPTION_OUTPUT_DIR = os.getenv("ISIC_MTACS_SAVED_CAPTIONS", "")  # Rỗng: dùng shared/caption_generation của run cũ.
ISIC_ROOT = os.getenv("ISIC_ROOT", "")
DINOV3_REPO_DIR = Path(os.getenv("DINOV3_REPO_DIR", "/kaggle/input/datasets/nguyenducxu/dinov3-code/dinov3-main"))
DINOV3_WEIGHTS = Path(os.getenv("DINOV3_WEIGHTS", "/kaggle/input/datasets/sinhnguync/dinov3-vitl16/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth"))
FLORENCE_MODEL_REF = os.getenv("FLORENCE_MODEL_REF", "/kaggle/input/notebooks/nguyenducxu/notebook-mt-moi/Florence-2-base")
CLIP_MODEL_REF = os.getenv("CLIP_MODEL_LOCAL_PATH", "/kaggle/input/datasets/nguyenducxu/vit-32/model_clip_vit")
OFFLINE_ASSETS_ROOT = os.getenv("FLORENCE_OFFLINE_ASSETS_ROOT", "/kaggle/input/notebooks/nguyenducxu/notebook-mt-moi")
PROJECT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT_DIR = Path(os.getenv("ISIC_MTACS_EXPORT_OUTPUT", str(PROJECT_DIR / "isic2017_mtacs_snapshot_web_export_v1")))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FEATURE_BATCH_SIZE = 4
RUN_FULL_TEST = False  # Bật nếu cần chạy đủ test của checkpoint; không cần cho demo.
COLD_RELOAD_DEVICE = DEVICE
CAPTION_VERIFY_DEVICE = DEVICE  # Caption và segmentation được kiểm lần lượt, không cùng giữ VRAM.
DOWNLOAD_PART_MIB = 256
print("Export output:", OUTPUT_DIR, "| device:", DEVICE)


In [ ]:
EXPORT_CODE_DIR = OUTPUT_DIR / "notebook_exporter"
EXPORT_CODE_DIR.mkdir(parents=True, exist_ok=True)
EXPORTERS = {'isic_export_only.py': '"""Recover an ISIC TACS checkpoint snapshot without training or changing its protocol."""\nfrom __future__ import annotations\n\nimport base64\nimport csv\nimport gc\nimport hashlib\nimport importlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nimport torch\n\nVERSION = "isic_tacs_snapshot_export_v1"\nRUNTIME_FILES = ("isic_segmentation_web.py", "tacs_visual_web.py", "train_isic_segmentation_web.py")\n\n\ndef sha(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(4 * 1024**2), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef object_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8-sig"))\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef source_identity(folder):\n    folder = Path(folder)\n    return object_hash({p.relative_to(folder).as_posix(): sha(p) for p in sorted(folder.rglob("*.py"))\n                        if not {".git", "__pycache__", ".cache"}.intersection(p.relative_to(folder).parts)})\n\n\ndef copy_verified(source, target):\n    source, target = Path(source), Path(target)\n    digest = sha(source)\n    if target.is_file():\n        if sha(target) != digest:\n            raise ValueError(f"Existing export artifact differs: {target}. Preserve it and select another OUTPUT_DIR.")\n        return\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_name(target.name + ".tmp")\n    shutil.copyfile(source, temporary)\n    if sha(temporary) != digest:\n        raise ValueError(f"Copy checksum mismatch: {source}")\n    temporary.replace(target)\n\n\ndef discover_run(reference="", roots=(Path("/kaggle/input"),), *, mode="tacs"):\n    explicit = str(reference).strip()\n    root = Path(explicit).expanduser() if explicit else None\n    if root is not None:\n        if not root.exists():\n            raise FileNotFoundError(f"SOURCE_RUN_DIR does not exist: {root}")\n        if root.is_file():\n            if root.name != "best.pt":\n                raise ValueError("SOURCE_RUN_DIR must be a saved run directory or best.pt")\n            root = root.parent\n        if (root / "best.pt").is_file():\n            return root.resolve()\n        roots = (root,)\n    found = []\n    for search in roots:\n        if not Path(search).is_dir():\n            continue\n        for current, dirs, files in os.walk(search):\n            dirs[:] = [name for name in dirs if name not in {".git", ".cache", "__pycache__", "images", "masks", "caption_runtime", "vendor"}]\n            if len(Path(current).relative_to(search).parts) >= 9:\n                dirs[:] = []\n            if {"best.pt", "protocol.json", "run_binding.json"}.issubset(files):\n                protocol = read_json(Path(current) / "protocol.json")\n                if protocol.get("config", {}).get("mode") == mode and protocol["config"].get("schema") == "isic2017_segmentation_web_v1":\n                    found.append(Path(current).resolve())\n    found = sorted(set(found))\n    if len(found) != 1:\n        raise FileNotFoundError(f"Set SOURCE_RUN_DIR to the exact saved ISIC {mode} run. Found: {found}")\n    return found[0]\n\n\ndef read_rows(path):\n    with Path(path).open(encoding="utf-8-sig", newline="") as handle:\n        rows = list(csv.DictReader(handle))\n    for row in rows:\n        if not re.fullmatch(r"ISIC_\\d+", row.get("image_id", "")):\n            raise ValueError(f"Unsafe or unexpected ISIC ID in {path}: {row.get(\'image_id\')}")\n        for key in ("image_sha256", "mask_sha256", "pixel_sha256"):\n            if not re.fullmatch(r"[0-9a-f]{64}", row.get(key, "")):\n                raise ValueError(f"Missing/invalid {key} for {row[\'image_id\']}")\n    if not rows or len({r["image_id"] for r in rows}) != len(rows):\n        raise ValueError(f"Empty/duplicate image IDs in {path}")\n    return rows\n\n\ndef validate_metadata(run, *, production=True, mode="tacs"):\n    """Uses meta storage: best/last optimizer tensors are never materialized here."""\n    run = Path(run)\n    for name in ("best.pt", "protocol.json", "run_binding.json", "candidate_pool.csv", "train_split.csv", "val_split.csv", "test_split.csv"):\n        if not (run / name).is_file():\n            raise FileNotFoundError(f"Missing saved sidecar: {run / name}")\n    protocol, binding = read_json(run / "protocol.json"), read_json(run / "run_binding.json")\n    checkpoint = torch.load(run / "best.pt", map_location="meta", weights_only=True)\n    if not {"state", "config", "identity", "epoch", "validation"}.issubset(checkpoint):\n        raise ValueError("Unsupported best.pt schema; do not rename keys or guess the model")\n    config = checkpoint["config"]\n    if (config != protocol.get("config") or config != binding.get("config")\n            or checkpoint["identity"] != protocol.get("identity") or object_hash(binding) != checkpoint["identity"]):\n        raise ValueError("best.pt / protocol / run_binding belong to different runs")\n    if mode not in {"tacs", "mtacs"} or config.get("mode") != mode or config.get("schema") != "isic2017_segmentation_web_v1" or config.get("context_top_k") != 1:\n        raise ValueError(f"Expected ISIC {mode} segmentation, K=1")\n    if production and (config.get("backbone") != "dinov3_vitl16" or config.get("feature_dim") != 1024 or config.get("image_size") != 448):\n        raise ValueError("Expected the saved ViT-L/16 448 segmentation architecture")\n    if not checkpoint["state"] or any(key.startswith("backbone.") for key in checkpoint["state"]):\n        raise ValueError("Expected learned segmentation/selector weights and a separate frozen DINO checkpoint")\n    frames = {role: read_rows(run / f"{role}_split.csv") for role in ("train", "val", "test")}\n    for role, rows in frames.items():\n        compact = [{key: row[key] for key in ("image_id", "image_sha256", "mask_sha256", "group_id")} for row in rows]\n        if compact != binding["splits"].get(role) or any(row["split"] != role for row in rows):\n            raise ValueError(f"Saved {role} split differs from the immutable training binding")\n    combined = sum(frames.values(), [])\n    if len({r["image_id"] for r in combined}) != len(combined):\n        raise ValueError("Saved splits overlap by image ID")\n    pixel_records = {}\n    for row in combined:\n        same = pixel_records.setdefault(row["pixel_sha256"], [])\n        if same and same[0]["split"] != row["split"]:\n            raise ValueError("Identical image pixels overlap across saved splits")\n        same.append(row)\n    pool = read_rows(run / "candidate_pool.csv")\n    train = {r["image_id"]: r for r in frames["train"]}\n    if [r["image_id"] for r in pool] != binding.get("pool_ids") or len(pool) < 2:\n        raise ValueError("Candidate order/identity changed")\n    if any(r != train.get(r["image_id"]) for r in pool):\n        raise ValueError("Candidates must be exact saved training records")\n    expected_count = round(len(train) * config["pool_fraction"])\n    if len(pool) != expected_count:\n        raise ValueError("The recorded candidate pool is not the saved 20% training pool")\n    history = []\n    if (run / "history.csv").is_file():\n        with (run / "history.csv").open(encoding="utf-8-sig", newline="") as handle:\n            history = list(csv.DictReader(handle))\n        epochs = [int(row["epoch"]) for row in history]\n        if epochs != list(range(1, len(history) + 1)):\n            raise ValueError("History epochs are incomplete/out of order")\n        selected = next((r for r in history if int(r["epoch"]) == int(checkpoint["epoch"])), None)\n        if selected is None or abs(float(selected["val_dice"]) - checkpoint["validation"]["dice"]) > 1e-7:\n            raise ValueError("Best validation score/epoch does not match history.csv")\n    last_meta = None\n    if (run / "last.pt").is_file():\n        last = torch.load(run / "last.pt", map_location="meta", weights_only=True)\n        if last.get("identity") != checkpoint["identity"]:\n            raise ValueError("last.pt belongs to another run")\n        last_meta = {key: last.get(key) for key in ("epoch", "bad_epochs", "best_dice")}\n        if history and int(last["epoch"]) != int(history[-1]["epoch"]):\n            raise ValueError("last.pt and history.csv have different completed epochs")\n        del last\n    completed = int(last_meta["epoch"]) if last_meta else int(history[-1]["epoch"]) if history else None\n    if completed is not None and completed < int(checkpoint["epoch"]):\n        raise ValueError("Best checkpoint is newer than the recorded training progress")\n    finished = None if completed is None else completed >= config["epochs"] or bool(\n        config.get("patience") and last_meta and last_meta["bad_epochs"] >= config["patience"])\n    snapshot = {"training_complete": finished, "requested_epochs": config["epochs"], "completed_epochs": completed,\n                "best_epoch": int(checkpoint["epoch"]), "validation_from_checkpoint": checkpoint["validation"],\n                "progress_source": "last.pt_meta" if last_meta else "history.csv" if history else "best_epoch_only",\n                "split_counts": {key: len(rows) for key, rows in frames.items()}, "pool_count": len(pool),\n                "repeated_pixels_within_split": [{"split": rows[0]["split"], "image_ids": [r["image_id"] for r in rows]}\n                    for rows in pixel_records.values() if len(rows) > 1],\n                "snapshot_note": "Use exact saved splits, including the original run\'s duplicate exclusions; do not resplit or resample."}\n    del checkpoint\n    return config, binding, frames, pool, snapshot\n\n\ndef stage_runtime(run, output, fallback, expected):\n    folder = Path(output) / "export_runtime"\n    folder.mkdir(parents=True, exist_ok=True)\n    original = Path(run) / "embedded_runtime"\n    supplied = sorted(p.name for p in original.glob("*.py")) if original.is_dir() else []\n    if supplied:\n        if source_identity(original) != expected:\n            raise ValueError("Original embedded_runtime no longer matches this trained run")\n        sources = {name: (original / name).read_bytes() for name in supplied}\n    else:\n        sources = {name: value.encode("utf-8") for name, value in fallback.items()}\n    if not set(RUNTIME_FILES).issubset(sources):\n        raise FileNotFoundError("Attach the original embedded_runtime files with best.pt")\n    if object_hash({name: hashlib.sha256(value).hexdigest() for name, value in sorted(sources.items())}) != expected:\n        raise ValueError("Embedded fallback is not the exact original runtime for this checkpoint")\n    for name, content in sources.items():\n        if Path(name).name != name or not name.endswith(".py"):\n            raise ValueError("Unsafe runtime filename")\n        compile(content, name, "exec")\n        path = folder / name\n        if path.exists() and path.read_bytes() != content:\n            raise ValueError("Export runtime changed; use a fresh OUTPUT_DIR")\n        if not path.exists():\n            path.write_bytes(content)\n    if source_identity(folder) != expected:\n        raise ValueError("Staged runtime contains unregistered source")\n    sys.path.insert(0, str(folder.resolve()))\n    for name in RUNTIME_FILES:\n        loaded = sys.modules.get(Path(name).stem)\n        if loaded and Path(loaded.__file__).resolve() != (folder / name).resolve():\n            raise RuntimeError("Another ISIC runtime is imported. Restart the kernel before running this notebook.")\n    net = importlib.import_module("isic_segmentation_web")\n    work = importlib.import_module("train_isic_segmentation_web")\n    return folder, net, work\n\n\ndef relocate_frames(frames, dataset, work, io):\n    dataset = work.resolve_dataset_root(dataset)\n    index = {}\n    for current, dirs, files in os.walk(dataset):\n        dirs[:] = [n for n in dirs if not n.startswith(\'.\')]\n        for name in files:\n            if Path(name).suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:\n                match = re.search(r"ISIC_\\d+", name, re.I)\n                if match:\n                    index.setdefault(match[0].upper(), []).append(Path(current) / name)\n    relocated, digests = {}, {}\n    def file_digest(path):\n        if path not in digests:\n            digests[path] = sha(path)\n        return digests[path]\n    for role, rows in frames.items():\n        result = []\n        for row in work.tqdm(rows, desc=f"Verify saved {role} images/masks"):\n            found = {}\n            for field, digest in (("image_path", row["image_sha256"]), ("mask_path", row["mask_sha256"])):\n                previous = Path(row[field])\n                candidates = ([previous] if previous.is_file() and previous.resolve().is_relative_to(dataset) else [])\n                candidates += sorted(index.get(row["image_id"], []))\n                unique = dict.fromkeys(p.resolve() for p in candidates)\n                match = next((p for p in unique if file_digest(p) == digest), None)\n                if match is None:\n                    raise FileNotFoundError(f"No content-matching {field} for {row[\'image_id\']} below {dataset}")\n                found[field] = str(match)\n            image = io.read_rgb(found["image_path"])\n            if work.pixel_hash(image) != row["pixel_sha256"]:\n                raise ValueError("Saved pixel identity mismatch: " + row["image_id"])\n            with Image.open(found["mask_path"]) as mask:\n                if mask.size != image.size or not set(np.unique(np.asarray(mask.convert("L"))).tolist()).issubset({0, 1, 255}):\n                    raise ValueError("Invalid binary mask/alignment: " + row["image_id"])\n            result.append(dict(row, **found))\n        relocated[role] = result\n    return dataset, relocated\n\n\ndef prepare(source_run, dataset_root, dino_repo, dino_weights, output, fallback_sources,\n            *, device="cuda", feature_batch_size=4, run_full_test=False, factory=None, mode="tacs"):\n    run = discover_run(source_run, mode=mode)\n    output = Path(output).expanduser().resolve()\n    if output == run or output.is_relative_to(run) or run.is_relative_to(output):\n        raise ValueError("OUTPUT_DIR must be separate from the read-only saved run")\n    if int(feature_batch_size) < 1:\n        raise ValueError("FEATURE_BATCH_SIZE must be positive")\n    config, binding, frames, pool, snapshot = validate_metadata(run, production=factory is None, mode=mode)\n    output.mkdir(parents=True, exist_ok=True)\n    runtime, net, work = stage_runtime(run, output, fallback_sources, binding["runtime"])\n    repo, weights = work.resolve_dino_assets(dino_repo, dino_weights, config) if factory is None else (Path(dino_repo).resolve(), Path(dino_weights).resolve())\n    if sha(weights) != binding["assets"]["backbone_sha256"] or source_identity(repo) != binding["assets"]["dino_source"]:\n        raise ValueError("DINO source/frozen pretrained weights differ from the trained checkpoint binding")\n    dataset, frames = relocate_frames(frames, dataset_root, work, net.io_utils)\n    by_id = {row["image_id"]: row for row in frames["train"]}\n    pool = [by_id[row["image_id"]] for row in pool]\n    best_sha = sha(run / "best.pt")\n    identity = {"exporter": VERSION if mode == "tacs" else "isic_mtacs_snapshot_export_v1", "run_identity": object_hash(binding), "source_best_sha256": best_sha,\n                "runtime_identity": binding["runtime"], "assets": binding["assets"],\n                "source_sidecars": {name: sha(run / name) for name in ("protocol.json", "run_binding.json", "candidate_pool.csv", "train_split.csv", "val_split.csv", "test_split.csv")},\n                "run_full_test": bool(run_full_test), "production": factory is None}\n    lock = output / "snapshot_binding.json"\n    if lock.exists() and read_json(lock) != identity:\n        raise ValueError("This export directory belongs to another checkpoint/options; choose a new OUTPUT_DIR")\n    write_json(lock, identity)\n    device = torch.device(device)\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA unavailable. Enable a GPU or select DEVICE=\'cpu\'.")\n    stage = output / f"bundle/isic2017_{mode}_vitl16_448_web_bundle"\n    write_json(output / "snapshot_summary.json", snapshot)\n    print(json.dumps(snapshot, indent=2))\n    return {"run": run, "output": output, "stage": stage, "runtime": runtime, "net": net, "work": work,\n            "config": config, "binding": binding, "identity": identity, "snapshot": snapshot,\n            "frames": frames, "pool": pool, "repo": repo, "weights": weights, "dataset": dataset,\n            "device": device, "feature_batch_size": int(feature_batch_size), "run_full_test": bool(run_full_test), "factory": factory}\n\n\ndef validate_bank(bank, pool, config, backbone_sha, prefix):\n    identity = object_hash({"backbone": backbone_sha, "image_size": config["image_size"],\n                           "pool": [(r["image_id"], r["image_sha256"]) for r in pool], "dtype": config["pool_token_dtype"]})\n    tokens, features = bank["tokens"], bank["features"]\n    if bank.get("identity") != identity or bank.get("image_ids") != [r["image_id"] for r in pool]:\n        raise ValueError("Candidate bank does not match the frozen backbone/preprocessing/original pool order")\n    if tokens.shape != (len(pool), prefix + (config["image_size"] // 16)**2, config["feature_dim"]) or features.shape != (len(pool), config["feature_dim"]):\n        raise ValueError("Candidate bank token/CLS shape mismatch")\n    if tokens.dtype != getattr(torch, config["pool_token_dtype"]) or features.dtype != torch.float32:\n        raise ValueError("Candidate bank precision differs from training")\n    for start in range(0, len(pool), 8):\n        if not torch.isfinite(tokens[start:start + 8]).all() or not torch.isfinite(features[start:start + 8]).all():\n            raise ValueError("Candidate bank contains nonfinite values")\n\n\ndef stage_manifest(ctx, status="staged", verification=None):\n    stage = ctx["stage"]\n    files = {p.relative_to(stage).as_posix(): sha(p) for p in sorted(stage.rglob("*"))\n             if p.is_file() and p.name not in {"manifest.json", "export_status.json"} and "__pycache__" not in p.parts and not p.name.endswith(\'.tmp\')}\n    manifest = {"schema": ctx["net"].SCHEMA, "mode": ctx["config"]["mode"], "status": status,\n                "backbone_file": "models/" + ctx["weights"].name, "caption_pixel_index": ctx.get("caption_pixel_index", {}),\n                "training_snapshot": ctx["snapshot"], "verification": verification or {}, "files": files}\n    write_json(stage / "manifest.json", manifest)\n    write_json(stage / "export_status.json", {"status": status, "training_complete": ctx["snapshot"]["training_complete"]})\n\n\ndef build(ctx):\n    """Restore best once, reuse/rebuild the frozen bank, and stage portable assets."""\n    net, work, config = ctx["net"], ctx["work"], ctx["config"]\n    if config["mode"] == "mtacs" and ("text_table" not in ctx or "stage_extra" not in ctx):\n        raise ValueError("M-TACS requires the caption-aware exporter and its original trained text cache")\n    stage, output = ctx["stage"], ctx["output"]\n    stage.mkdir(parents=True, exist_ok=True)\n    # Any retry revalidates the export instead of trusting an old ready marker.\n    (stage / "manifest.json").unlink(missing_ok=True)\n    write_json(stage / "export_status.json", {"status": "staging"})\n    print("Restoring the original frozen backbone + all learned best.pt parameters (no training)")\n    model = (ctx["factory"](config) if ctx["factory"] else net.build_model(ctx["repo"], config, ctx["weights"])).to(ctx["device"]).eval()\n    saved = torch.load(ctx["run"] / "best.pt", map_location="cpu", weights_only=True)\n    if saved["identity"] != ctx["identity"]["run_identity"] or sha(ctx["run"] / "best.pt") != ctx["identity"]["source_best_sha256"]:\n        raise ValueError("best.pt changed after preflight")\n    model.restore_trainable(saved["state"])\n    is_selector = lambda key: key.startswith("selector.") or key == "semantic_mix_logit"\n    for name, chosen in (("selector.pt", True), ("segmentor.pt", False)):\n        work.atomic_torch(stage / "models" / name, {k: v for k, v in saved["state"].items() if is_selector(k) == chosen})\n    del saved\n    copy_verified(ctx["weights"], stage / "models" / ctx["weights"].name)\n    bank_path = ctx["run"] / "shared/candidate_bank.pt"\n    if bank_path.is_file():\n        print("Reusing the original frozen candidate token/CLS bank")\n        bank = torch.load(bank_path, map_location="cpu", weights_only=True, mmap=True)\n    else:\n        bank_path = output / "rebuilt_candidate_bank.pt"\n        print("Rebuilding only the frozen candidate bank from the SAME saved pool")\n        bank = work.build_bank(model, ctx["pool"], dict(config, feature_batch_size=ctx["feature_batch_size"]), bank_path,\n                               ctx["binding"]["assets"]["backbone_sha256"])\n    validate_bank(bank, ctx["pool"], config, ctx["binding"]["assets"]["backbone_sha256"], model.prefix)\n    copy_verified(bank_path, stage / "candidate_bank.pt")\n    write_json(stage / "config.json", config)\n    for p in ctx["runtime"].glob("*.py"):\n        copy_verified(p, stage / "runtime" / p.name)\n    shutil.copytree(ctx["repo"], stage / "vendor/dinov3", dirs_exist_ok=True,\n        ignore=shutil.ignore_patterns(".git", ".cache", "__pycache__", "*.pyc", "*.pth", "*.pt", "*.ipynb"))\n    if source_identity(stage / "vendor/dinov3") != ctx["binding"]["assets"]["dino_source"]:\n        raise ValueError("Exported DINO source differs from the saved source identity")\n    exported_pool = []\n    for position, row in enumerate(ctx["pool"]):\n        path = stage / "candidate_images" / (row["image_id"] + Path(row["image_path"]).suffix.lower())\n        copy_verified(row["image_path"], path)\n        exported_pool.append({"image_id": row["image_id"], "position": position, "image": path.relative_to(stage).as_posix(),\n                              "pixel_sha256": row["pixel_sha256"], "group_id": row["group_id"], "split": "train"})\n    write_json(stage / "candidate_pool.json", exported_pool)\n    gt, pixel_groups = {}, {}\n    for role, rows in ctx["frames"].items():\n        portable = []\n        for row in work.tqdm(rows, desc=f"Export {role} ground-truth masks"):\n            path = stage / "ground_truth_masks" / (row["image_id"] + ".png")\n            path.parent.mkdir(parents=True, exist_ok=True)\n            with Image.open(row["mask_path"]) as image:\n                Image.fromarray((np.asarray(image.convert("L")) > 0).astype(np.uint8) * 255).save(path)\n            gt[row["pixel_sha256"]] = {"image_id": row["image_id"], "split": role, "mask": path.relative_to(stage).as_posix(), "group_id": row["group_id"]}\n            pixel_groups.setdefault(row["pixel_sha256"], []).append(row)\n            portable.append({k: v for k, v in row.items() if k not in {"image_path", "mask_path"}})\n        work.csv_write(stage / f"{role}_split.csv", portable)\n    ambiguities = []\n    for digest, rows in pixel_groups.items():\n        if len(rows) < 2:\n            continue\n        groups = {row["group_id"] for row in rows}\n        masks = {sha(stage / "ground_truth_masks" / (row["image_id"] + ".png")) for row in rows}\n        record = {"pixel_sha256": digest, "split": rows[0]["split"], "image_ids": [r["image_id"] for r in rows]}\n        if len(masks) > 1:\n            if any(groups):\n                raise ValueError("Conflicting ground-truth aliases with group metadata need a group-aware ambiguity adapter")\n            gt.pop(digest)\n            record["automatic_ground_truth"] = "omitted_conflicting_masks"\n        else:\n            if len(groups) > 1:\n                raise ValueError("Identical-pixel aliases have conflicting group metadata")\n            gt[digest]["image_id"] = None\n            gt[digest]["image_ids"] = record["image_ids"]\n            record["automatic_ground_truth"] = "same_mask_image_id_ambiguous"\n        ambiguities.append(record)\n    # Preserve every recorded test row and mask. Ambiguous uploaded pixels may\n    # not silently pick one of several different annotations as ground truth.\n    write_json(stage / "ground_truth_ambiguities.json", ambiguities)\n    write_json(stage / "ground_truth_index.json", gt)\n    source_folder = stage / "provenance/original_run"\n    for name in ("protocol.json", "run_binding.json", "candidate_pool.csv", "train_split.csv", "val_split.csv", "test_split.csv", "history.csv"):\n        if (ctx["run"] / name).is_file():\n            copy_verified(ctx["run"] / name, source_folder / name)\n    write_json(stage / "training_snapshot.json", ctx["snapshot"])\n    write_json(stage / "export_provenance.json", ctx["identity"])\n    write_json(stage / "attention_config.json", {"method": net.ATTENTION_METHOD, "grid": [config["image_size"] // 16]*2,\n        "roles": ["query_segmentor", "candidate_selector"], "separate_attention_model": False,\n        "explanation": "Last-block mean-head CLS-to-patch self-attention, not Grad-CAM or causal attribution"})\n    if ctx.get("stage_extra"):\n        ctx["stage_extra"](ctx)\n    metrics = {"dataset": "ISIC2017", "task": "binary_segmentation", "mode": config["mode"], "threshold": config["threshold"],\n        "validation_from_checkpoint": ctx["snapshot"]["validation_from_checkpoint"], "best_epoch": ctx["snapshot"]["best_epoch"],\n        "training_complete": ctx["snapshot"]["training_complete"], "completed_epochs": ctx["snapshot"]["completed_epochs"],\n        "split_counts": ctx["snapshot"]["split_counts"], "test": None, "test_status": "not_run_export_only"}\n    if ctx["run_full_test"]:\n        values, rows = work.evaluate(model, work.make_loader(ctx["frames"]["test"], dict(config, workers=0, eval_batch_size=1)),\n            ctx["pool"], bank, config, table=ctx.get("text_table"), desc="Optional test / recovered best checkpoint")\n        metrics.update(test=values, test_status="evaluated_snapshot", aggregation="mean_per_image_at_network_resolution")\n        work.csv_write(stage / "test_predictions.csv", rows)\n    write_json(stage / "metrics.json", metrics)\n    example = ctx.get("example_record", ctx["frames"]["val"][0])\n    example_path = stage / "example" / ("query" + Path(example["image_path"]).suffix.lower())\n    copy_verified(example["image_path"], example_path)\n    with torch.no_grad():\n        initial = model.prepare(net.io_utils.preprocess(example["image_path"], config)[None].to(ctx["device"]))\n        global_query, _ = model.encode_selector_tokens(initial)\n        allowed = work.allowed_candidates([example], ctx["pool"], ctx["device"])\n        q_text = q_valid = c_text = c_valid = None\n        if ctx.get("text_table") is not None:\n            q_text, q_valid = ctx["text_table"].gather([example], ctx["device"])\n            c_text, c_valid = ctx["text_table"].gather(ctx["pool"], ctx["device"])\n        scores = model.score(global_query, bank["features"].to(ctx["device"]), allowed, q_text, q_valid, c_text, c_valid)\n        position = int(scores.argmax(1))\n        logits, _ = model.decode(initial, bank["tokens"][position:position + 1].to(ctx["device"], dtype=torch.float32),\n            q_text, q_valid, c_text[position:position + 1] if c_text is not None else None,\n            c_valid[position:position + 1] if c_valid is not None else None)\n    work.atomic_torch(stage / "example/expected_logits.pt", logits.detach().cpu())\n    write_json(stage / "example/reference.json", {"context_position": position, "context_id": ctx["pool"][position]["image_id"],\n        "query_id": example["image_id"], "checkpoint_sha256": ctx["identity"]["source_best_sha256"]})\n    if not (stage / "requirements.txt").exists():\n        (stage / "requirements.txt").write_text("torch>=2.5\\nnumpy\\nPillow\\ntqdm\\n", encoding="utf-8")\n    ctx.update(example_path=example_path)\n    del model, bank, logits, initial, global_query, allowed, scores, q_text, q_valid, c_text, c_valid\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    stage_manifest(ctx)\n    print("All model/data artifacts staged. Ready will be set only after independent reload + attention/logit verification.")\n    return ctx\n\n\nCOLD_SOURCE = r\'\'\'import base64, json, sys\nfrom pathlib import Path\nimport numpy as np\nimport torch\ntorch.set_num_threads(2)\ntorch.set_float32_matmul_precision("highest")\ntorch.backends.cuda.matmul.allow_tf32 = False\ntorch.backends.cudnn.allow_tf32 = False\nroot, image, device = Path(sys.argv[1]).resolve(), Path(sys.argv[2]).resolve(), sys.argv[3]\nsys.path.insert(0, str(root / "runtime"))\nimport isic_segmentation_web as net\nprint("Cold loading original DINO + selector.pt + segmentor.pt", flush=True)\npredictor = net.Predictor(root, device, _allow_staged=True)\nplain = predictor.predict(image, return_attention=False)\nexplained = predictor.predict(image, return_attention=True)\nexpected = torch.load(root / "example/expected_logits.pt", map_location="cpu", weights_only=True)\nreference = net.io_utils.read_json(root / "example/reference.json")\nif not torch.allclose(plain["logits"], explained["logits"], atol=1e-5, rtol=1e-5):\n    raise ValueError("Attention capture changed logits")\nif not torch.allclose(expected, plain["logits"], atol=5e-4, rtol=5e-4):\n    raise ValueError("Split exported weights differ from the loaded best checkpoint")\ncandidate = explained["evidence"]["candidate"]\nif candidate["position"] != reference["context_position"] or candidate["image_id"] != reference["context_id"]:\n    raise ValueError("Export changed the selected context")\nfor role in ["query_segmentor", "candidate_selector"]:\n    attention = explained["evidence"]["attention"][role]\n    if attention["grid_shape"] != [predictor.config["image_size"]//16]*2 or not np.isfinite(attention["grid"]).all():\n        raise ValueError("Missing/invalid attention grid")\n    (root / "example" / (role + ".png")).write_bytes(base64.b64decode(attention["image_url"].split(",",1)[1]))\ntorch.save(explained.pop("logits"), root / "example/reloaded_logits.pt")\nnp.save(root / "example/probabilities.npy", explained.pop("probabilities"))\nfor role in ["mask", "overlay"]:\n    (root / "example" / (role + ".png")).write_bytes(base64.b64decode(explained["prediction"][role + "_url"].split(",",1)[1]))\nnet.io_utils.write_json(root / "example/prediction.json", explained)\nnet.io_utils.write_json(root / "smoke_test.json", {"state_reload":True, "trained_model_parity":True,\n    "attention_parity":True, "selected_context_parity":True, "cold_offline_reload":True,\n    "device":device, "benchmark_metric":False})\nprint("Independent offline checkpoint/attention reload passed", flush=True)\n\'\'\'\n\n\ndef verify_and_publish(ctx, *, device=None):\n    stage = ctx["stage"]\n    verification_device = str(device or ctx["device"])\n    if ctx["factory"] is None:\n        script = ctx["output"] / "verify_isic_snapshot.py"\n        script.write_text(ctx.get("cold_source", COLD_SOURCE), encoding="utf-8")\n        env = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")\n        env.pop("PYTHONPATH", None)\n        command = [sys.executable, str(script), str(stage), str(ctx["example_path"]), verification_device]\n        log_path = ctx["output"] / "cold_reload.log"\n        print(f"Independent offline reload on {verification_device}; progress/error log: {log_path}", flush=True)\n        with log_path.open("w", encoding="utf-8") as log:\n            result = subprocess.run(command, cwd=stage, env=env, stdout=log, stderr=subprocess.STDOUT, timeout=1800)\n        if result.returncode:\n            raise RuntimeError("Cold reload failed; staged artifacts remain available.\\n" + log_path.read_text(\'utf-8\')[-5000:])\n        print(log_path.read_text("utf-8")[-1600:])\n        smoke = read_json(stage / "smoke_test.json")\n        status = "ready"\n    else:\n        # Small explicit fixtures can test the real exporter; they are never deployable models.\n        predictor = ctx["net"].Predictor(stage, "cpu", factory=ctx["factory"], _allow_staged=True)\n        plain = predictor.predict(ctx["example_path"], return_attention=False)\n        explained = predictor.predict(ctx["example_path"], return_attention=True)\n        reference = read_json(stage / "example/reference.json")\n        expected = torch.load(stage / "example/expected_logits.pt", map_location="cpu", weights_only=True)\n        if (not torch.allclose(expected, plain["logits"], atol=5e-4, rtol=5e-4)\n                or not torch.allclose(plain["logits"], explained["logits"], atol=1e-5, rtol=1e-5)\n                or explained["evidence"]["candidate"]["position"] != reference["context_position"]):\n            raise ValueError("Fixture checkpoint/attention reload failed")\n        explained.pop("logits")\n        np.save(stage / "example/probabilities.npy", explained.pop("probabilities"))\n        write_json(stage / "example/prediction.json", explained)\n        smoke = {"state_reload": True, "trained_model_parity": True, "attention_parity": True,\n                 "cold_offline_reload": False, "benchmark_metric": False, "fixture": True}\n        write_json(stage / "smoke_test.json", smoke)\n        status = "fixture_unverified"\n    stage_manifest(ctx, status, smoke)\n    write_json(ctx["output"] / "export_receipt.json", {"status": status, "bundle": str(stage),\n        "source_best_sha256": ctx["identity"]["source_best_sha256"], "training_snapshot": ctx["snapshot"]})\n    return stage\n\n\nclass _SplitArchive:\n    """Unseekable ZIP sink: never creates an additional multi-GB full ZIP."""\n    def __init__(self, folder, name, limit):\n        self.folder, self.name, self.limit = Path(folder), name, limit\n        self.offset, self.handle, self.current_size = 0, None, 0\n        self.files, self.digest = [], hashlib.sha256()\n\n    def seekable(self):\n        return False\n\n    def tell(self):\n        return self.offset\n\n    def write(self, data):\n        data = memoryview(data)\n        written = len(data)\n        self.digest.update(data)\n        while data:\n            if self.handle is None or self.current_size == self.limit:\n                if self.handle:\n                    self.handle.close()\n                path = self.folder / f"{self.name}.part{len(self.files) + 1:03d}"\n                self.files.append(path)\n                self.handle, self.current_size = path.open("wb"), 0\n            size = min(len(data), self.limit - self.current_size)\n            self.handle.write(data[:size])\n            self.current_size += size\n            self.offset += size\n            data = data[size:]\n        return written\n\n    def flush(self):\n        if self.handle:\n            self.handle.flush()\n\n    def close(self):\n        if self.handle:\n            self.handle.close()\n            self.handle = None\n\n\nJOIN_SOURCE = r\'\'\'from pathlib import Path\nimport hashlib, json, sys\nfolder = Path(__file__).resolve().parent\ninfo = json.loads((folder / "download_manifest.json").read_text("utf-8"))\ntarget = Path(sys.argv[1]) if len(sys.argv)>1 else folder / info["archive_name"]\nif target.exists():\n    raise FileExistsError("Preserve the existing ZIP or choose another output: " + str(target))\ntemporary = target.with_name(target.name + ".tmp")\nfull = hashlib.sha256()\nwith temporary.open("wb") as out:\n    for item in info["parts"]:\n        path = folder / item["name"]\n        if path.parent != folder or path.stat().st_size != item["bytes"]:\n            raise ValueError("Missing/wrong archive part: " + item["name"])\n        part = hashlib.sha256()\n        with path.open("rb") as src:\n            for block in iter(lambda:src.read(4*1024**2), b""):\n                part.update(block); full.update(block); out.write(block)\n        if part.hexdigest() != item["sha256"]:\n            raise ValueError("Corrupt part: " + item["name"])\nif full.hexdigest() != info["archive_sha256"]:\n    raise ValueError("Joined archive checksum mismatch")\ntemporary.replace(target)\nprint("Verified ZIP:", target)\nprint("Extract it to obtain the complete ISIC web bundle.")\n\'\'\'\n\n\ndef package_downloads(ctx, *, part_mib=256):\n    stage, output = ctx["stage"], ctx["output"]\n    if read_json(stage / "manifest.json")["status"] != "ready":\n        raise ValueError("Only a verified production bundle may be packaged for deployment")\n    ctx["net"].verify_bundle(stage)\n    if type(part_mib) is not int or part_mib < 16:\n        raise ValueError("DOWNLOAD_PART_MIB must be an integer >=16")\n    folder = output / "downloads"\n    folder.mkdir(parents=True, exist_ok=True)\n    (folder / "download_manifest.json").unlink(missing_ok=True)\n    archive_name = stage.name + ".zip"\n    sink = _SplitArchive(folder, archive_name, part_mib * 1024**2)\n    try:\n        with zipfile.ZipFile(sink, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:\n            for path in sorted(stage.rglob("*")):\n                if path.is_file() and "__pycache__" not in path.parts and not path.name.endswith(".tmp"):\n                    archive.write(path, path.relative_to(stage.parent).as_posix())\n    finally:\n        sink.close()\n    parts = [{"name": p.name, "bytes": p.stat().st_size, "sha256": sha(p)} for p in sink.files]\n    write_json(folder / "download_manifest.json", {"archive_name": archive_name, "archive_sha256": sink.digest.hexdigest(),\n        "part_size_mib": part_mib, "parts": parts, "bundle": stage.name})\n    (folder / "join_downloads.py").write_text(JOIN_SOURCE, encoding="utf-8")\n    print(f"Download {len(parts)} ZIP parts, download_manifest.json and join_downloads.py from {folder}")\n    print("On Windows: python join_downloads.py (checks hashes and joins the ZIP; no Torch required).")\n    return folder\n', 'isic_mtacs_export_only.py': '"""Export an interrupted full_both ISIC run using its immutable captions/cache.\n\nNo training, caption regeneration or embedding replacement is performed. The\nonly new caption is a forced fresh-image deployment check in an offline child.\n"""\nfrom __future__ import annotations\n\nimport importlib\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\n\nimport torch\n\nimport isic_export_only as common\n\n\ndef caption_directory(run, reference, assets):\n    candidates = [Path(reference).expanduser()] if str(reference).strip() else [Path(run) / "shared/caption_generation"]\n    if not str(reference).strip():\n        for current, dirs, files in os.walk(Path(run) / "shared"):\n            dirs[:] = [n for n in dirs if n not in {"generated", "caption_runtime", "__pycache__"}]\n            if "web_caption_settings.json" in files:\n                candidates.append(Path(current))\n    matches = []\n    required = {"captions.csv": "caption_csv_sha256", "text_cache/clip_text.pt": "text_cache_sha256",\n                "web_caption_settings.json": "caption_settings_sha256"}\n    for path in sorted(set(p.resolve() for p in candidates)):\n        if all((path / name).is_file() and common.sha(path / name) == assets.get(key) for name, key in required.items()):\n            matches.append(path)\n    if len(matches) != 1:\n        raise FileNotFoundError("Set CAPTION_OUTPUT_DIR to the ORIGINAL shared/caption_generation folder. "\n            "Its captions.csv, text_cache/clip_text.pt and web_caption_settings.json must match run_binding.json. "\n            f"Matching folders: {matches}. Do not regenerate captions for this checkpoint.")\n    return matches[0]\n\n\ndef registered_asset(reference, expected, kind, core, roots):\n    """Find exactly the model files registered at training, never a substitute."""\n    if not expected or any(Path(n).name != n or "\\\\" in n for n in expected):\n        raise ValueError(f"Invalid registered {kind} asset names")\n    visited, candidates = set(), []\n    explicit = Path(str(reference)).expanduser() if str(reference).strip() else None\n    if explicit is not None and explicit.is_dir():\n        candidates.append(explicit)\n    for root in roots:\n        root = Path(root)\n        if not root.is_dir():\n            continue\n        for current, dirs, files in os.walk(root):\n            dirs[:] = [n for n in dirs if n not in {".git", ".cache", "__pycache__", "legacy_packages", "hf_modules",\n                        "images", "masks", "generated", "vendor", "candidate_images", "ground_truth_masks"}]\n            if len(Path(current).relative_to(root).parts) >= 10:\n                dirs[:] = []\n            if "config.json" in files:\n                candidates.append(Path(current))\n    failures = []\n    for path in candidates:\n        path = path.resolve()\n        if path in visited:\n            continue\n        visited.add(path)\n        try:\n            cfg = common.read_json(path / "config.json")\n        except (OSError, ValueError):\n            continue\n        if cfg.get("model_type") != ("florence2" if kind == "florence" else "clip"):\n            continue\n        if any(not (path / name).is_file() for name in expected):\n            continue\n        if core.checkpoint_asset_identity(path) == expected:\n            return path\n        failures.append(str(path))\n    raise FileNotFoundError(f"Attach the SAME complete local {kind} model used to create the saved captions. "\n        f"Registered files/hashes do not match any checkpoint. Set {kind.upper()}_MODEL_REF explicitly. Checked: {failures}")\n\n\ndef validate_captions(ctx, reference=""):\n    core = importlib.import_module("isic_caption_core")\n    folder = caption_directory(ctx["run"], reference, ctx["binding"]["assets"])\n    required = ("status.json", "generation_protocol/protocol.json", "text_cache/protocol.json")\n    for name in required:\n        if not (folder / name).is_file():\n            raise FileNotFoundError(f"Missing original caption provenance: {folder / name}")\n    registered = common.read_json(folder / "web_caption_settings.json")\n    generation = common.read_json(folder / "generation_protocol/protocol.json")\n    text_protocol = common.read_json(folder / "text_cache/protocol.json")\n    records = [{key: row[key] for key in ("image_id", "image_path", "split", "image_sha256")}\n               for role in ("train", "val", "test") for row in ctx["frames"][role]]\n    rows, status = core.validated_export({"output": str(folder), "records": records})\n    images = [{key: row[key] for key in ("image_id", "split", "image_sha256")} for row in records]\n    code_sha = common.sha(ctx["runtime"] / "isic_caption_core.py")\n    settings = registered.get("settings", {})\n    if (registered.get("policy") != core.POLICY or registered.get("method") != core.METHOD\n            or registered.get("model_id") != core.MODEL_ID or settings.get("precision") != "float32"\n            or generation.get("dtype") != "float32" or generation.get("settings") != settings\n            or generation.get("method") != core.METHOD or generation.get("model_id") != core.MODEL_ID\n            or generation.get("policy") != core.POLICY or generation.get("images") != images\n            or generation.get("code_sha256") != code_sha or common.object_hash(generation) != status["identity"]\n            or generation.get("model_assets") != registered.get("florence_assets")):\n        raise ValueError("Original caption generation/settings/code/image binding is inconsistent; do not replace it")\n    if (text_protocol.get("caption_csv_sha256") != status["csv_sha256"]\n            or text_protocol.get("clip_assets") != registered.get("clip_assets")\n            or text_protocol.get("policy") != core.POLICY or text_protocol.get("code_sha256") != code_sha):\n        raise ValueError("CLIP text protocol does not match this original caption CSV/code/checkpoint")\n    cache_path = folder / "text_cache/clip_text.pt"\n    table = ctx["work"].TextTable(cache_path)\n    core.validate_text_cache(table.data, rows, status["csv_sha256"], common.object_hash(text_protocol))\n    if table.data["embeddings"].dtype != torch.float32 or table.data["embeddings"].shape != (len(rows), ctx["config"]["text_dim"]):\n        raise ValueError("Expected the original FP32 512-dimensional CLIP text table")\n    if ctx["config"]["text_dim"] != 512:\n        raise ValueError("This full_both deployment uses CLIP ViT-B/32 text_dim=512")\n    coverage = {}\n    for role, frame in [*ctx["frames"].items(), ("candidate_pool", ctx["pool"])]:\n        _, valid = table.gather(frame, "cpu")\n        coverage[role] = {"images": len(frame), "valid": int(valid.sum())}\n    if not coverage["train"]["valid"] or not coverage["candidate_pool"]["valid"]:\n        raise ValueError("The saved full_both run has no usable query/candidate training text")\n    ctx.update(caption_root=folder, caption_registered=registered, caption_rows=rows, text_table=table,\n               caption_core=core, caption_coverage=coverage)\n    # Prefer a usable validation caption for the cached path parity test.\n    ctx["example_record"] = next((r for r in ctx["frames"]["val"] if bool(table.data["valid"][table.positions[r["image_id"]]])), ctx["frames"]["val"][0])\n    return ctx\n\n\ndef prepare(source_run, dataset_root, dino_repo, dino_weights, output, fallback_sources,\n            *, caption_output="", florence_ref="", clip_ref="", offline_assets="", device="cuda",\n            feature_batch_size=4, run_full_test=False, factory=None):\n    ctx = common.prepare(source_run, dataset_root, dino_repo, dino_weights, output, fallback_sources,\n        mode="mtacs", device=device, feature_batch_size=feature_batch_size, run_full_test=run_full_test, factory=factory)\n    for name in ("isic_caption_core", "isic_caption_web"):\n        loaded = sys.modules.get(name)\n        if loaded and Path(loaded.__file__).resolve() != ctx["runtime"] / (name + ".py"):\n            raise RuntimeError("Another caption runtime is imported. Restart the kernel and Run All.")\n    validate_captions(ctx, caption_output)\n    roots = [Path(offline_assets), ctx["run"], Path("/kaggle/input")]\n    core = ctx["caption_core"]\n    asset_paths = {key: registered_asset(ref, ctx["caption_registered"][key + "_assets"], key, core, roots)\n                   for key, ref in (("florence", florence_ref), ("clip", clip_ref))}\n    if factory is None:\n        core.validate_florence_prefix_folder(asset_paths["florence"])\n        importlib.import_module("isic_caption_web").validate_clip_assets(asset_paths["clip"])\n    ctx.update(caption_asset_paths=asset_paths, offline_assets=offline_assets, stage_extra=stage_captions,\n               cold_source=COLD_SOURCE)\n    common.write_json(ctx["output"] / "caption_coverage.json", ctx["caption_coverage"])\n    print("Original captions and text cache verified:", ctx["caption_root"])\n    print("Caption coverage:", ctx["caption_coverage"])\n    return ctx\n\n\ndef stage_wheels(ctx):\n    core = ctx["caption_core"]\n    destination = ctx["stage"] / "offline_wheels"\n    roots = [ctx["caption_root"] / "caption_runtime", Path(ctx["offline_assets"]), Path("/kaggle/input")]\n    dirs = core.offline_caption_wheels(roots, ctx["output"] / "collected_wheels")\n    prefixes = ("transformers-", "tokenizers-", "huggingface_hub-", "einops-", "timm-", "accelerate-")\n    wheel_map = {}\n    for folder in dirs:\n        for path in sorted(Path(folder).glob("*.whl")):\n            if path.name.startswith(prefixes):\n                common.copy_verified(path, destination / path.name)\n                wheel_map[path.name] = common.sha(path)\n    if ctx["factory"] is None and any(not any(n.startswith(prefix) for n in wheel_map)\n            for prefix in ("transformers-4.49.", "tokenizers-0.21.", "huggingface_hub-0.")):\n        raise FileNotFoundError("Attach offline wheels for transformers 4.49.x, tokenizers 0.21.x and huggingface-hub<1 "\n            "using OFFLINE_ASSETS_ROOT. The old caption_runtime/wheels can also be reused. "\n            "Installed legacy_packages are platform-specific and are not copied into the web bundle.")\n    common.write_json(ctx["stage"] / "offline_wheels_manifest.json", wheel_map)\n\n\ndef stage_captions(ctx):\n    stage, folder = ctx["stage"], ctx["caption_root"]\n    for key, path in ctx["caption_asset_paths"].items():\n        shutil.copytree(path, stage / "caption_models" / key, dirs_exist_ok=True,\n            ignore=shutil.ignore_patterns(".git", ".cache", "__pycache__", "*.pyc"))\n        if ctx["caption_core"].checkpoint_asset_identity(stage / "caption_models" / key) != ctx["caption_registered"][key + "_assets"]:\n            raise ValueError("Copied caption model files differ from their trained asset binding")\n    for original, target in (("captions.csv", "captions.csv"), ("web_caption_settings.json", "caption_settings.json"),\n                             ("text_cache/clip_text.pt", "text_embeddings.pt")):\n        common.copy_verified(folder / original, stage / target)\n    for name in ("generation_protocol/protocol.json", "text_cache/protocol.json", "text_cache/status.json", "status.json"):\n        if (folder / name).is_file():\n            common.copy_verified(folder / name, stage / "caption_provenance" / name)\n    common.write_json(stage / "caption_lookup.json", {r["image_id"]: r for r in ctx["caption_rows"]})\n    common.write_json(stage / "caption_coverage.json", ctx["caption_coverage"])\n    pixels, index, ambiguous = {}, {}, []\n    for rows in ctx["frames"].values():\n        for row in rows:\n            pixels.setdefault(row["pixel_sha256"], []).append(row["image_id"])\n    lookup = {r["image_id"]: r for r in ctx["caption_rows"]}\n    table = ctx["text_table"]\n    for digest, ids in pixels.items():\n        first = table.positions[ids[0]]\n        equal = all(lookup[identity]["final_caption"] == lookup[ids[0]]["final_caption"]\n                    and torch.equal(table.data["embeddings"][table.positions[identity]], table.data["embeddings"][first])\n                    and bool(table.data["valid"][table.positions[identity]]) == bool(table.data["valid"][first]) for identity in ids)\n        if equal:\n            index[digest] = ids[0]\n        else:\n            ambiguous.append({"pixel_sha256": digest, "image_ids": ids, "action": "generate_fresh_caption_for_uploaded_pixels"})\n    ctx["caption_pixel_index"] = index\n    common.write_json(stage / "caption_pixel_ambiguities.json", ambiguous)\n    stage_wheels(ctx)\n    (stage / "requirements.txt").write_text("torch>=2.5\\nnumpy\\nPillow\\ntqdm\\n"\n        "# Caption child (4.49.x), isolated from the segmentor kernel:\\ntransformers==4.49.0\\ntokenizers>=0.21,<0.22\\n"\n        "huggingface-hub>=0.26,<1\\naccelerate\\neinops\\ntimm\\n", encoding="utf-8")\n\n\nFRESH_SOURCE = r\'\'\'import hashlib, sys\nfrom pathlib import Path\nimport numpy as np\nimport torch\ntorch.set_num_threads(2)\ntorch.set_float32_matmul_precision("highest")\ntorch.backends.cuda.matmul.allow_tf32=False\ntorch.backends.cudnn.allow_tf32=False\nroot, device=Path(sys.argv[1]).resolve(), sys.argv[2]\nsys.path.insert(0,str(root/\'runtime\'))\nimport isic_caption_core as core\nfrom isic_caption_web import _LocalCaptionEngine\nfrom tacs_visual_web import read_json, write_json, read_rgb\nreference=read_json(root/\'example/reference.json\')\nimages=list((root/\'example\').glob(\'query.*\'))\nimage=next(p for p in images if p.suffix.lower() in {\'.png\',\'.jpg\',\'.jpeg\',\'.bmp\'})\nregistered=read_json(root/\'caption_settings.json\')\ntorch.manual_seed(registered[\'settings\'][\'seed\'])\nengine=_LocalCaptionEngine(root,device)\nrow,vector,valid=engine.caption(image)  # Forced generation: bypass the pixel/CSV cache.\nif vector.shape!=(512,) or vector.dtype!=torch.float32 or not torch.isfinite(vector).all():\n    raise ValueError(\'Fresh caption embedding has invalid shape/values\')\nif (not bool(valid) and vector.count_nonzero()) or (bool(valid) and abs(float(vector.norm())-1)>1e-4):\n    raise ValueError(\'Fresh caption mask/normalization is invalid\')\nfinal,approved,clinical,words=core.caption_policy(row[\'raw_caption\'])\n# Token-limit rejection can make a structurally valid caption invalid; it must never become valid otherwise.\nif row[\'final_caption\']!=final or (bool(valid) and not approved):\n    raise ValueError(\'Fresh caption does not obey the saved policy\')\ncache=torch.load(root/\'text_embeddings.pt\',map_location=\'cpu\',weights_only=True)\nlookup=read_json(root/\'caption_lookup.json\')\ni=next(i for i,v in enumerate(cache[\'valid\']) if bool(v))\ntext=lookup[cache[\'image_ids\'][i]][\'final_caption\']\ntokens=engine.tokenizer([text],padding=True,truncation=True,\n    max_length=engine.clip.config.max_position_embeddings,return_tensors=\'pt\')\nwith torch.inference_mode():\n    encoded=torch.nn.functional.normalize(engine.clip(**{k:v.to(engine.device) for k,v in tokens.items()}).text_embeds.float(),dim=1)[0].cpu()\nif not torch.allclose(encoded,cache[\'embeddings\'][i],atol=5e-4,rtol=5e-4):\n    raise ValueError(\'Reloaded CLIP does not reproduce the original trained text embedding\')\nrgb=read_rgb(image)\npixel_hash=hashlib.sha256(np.asarray(rgb).tobytes()+str(rgb.size).encode()).hexdigest()\ntorch.save({\'row\':row,\'vector\':vector.clone(),\'valid\':valid.clone(),\n    \'pixel_sha256\':pixel_hash},root/\'example/fresh_caption.pt\')\nwrite_json(root/\'example/fresh_caption_check.json\',{\'forced_fresh_generation\':True,\'florence_clip_reload\':True,\n    \'original_clip_embedding_parity\':True,\'device\':device,\'fresh_caption\':row,\'valid\':bool(valid),\n    \'embedding_shape\':list(vector.shape),\'benchmark_metric\':False})\nprint(\'Actual exported Florence + CLIP fresh-image check passed\',flush=True)\n\'\'\'\n\n\n# The shared script verifies exact checkpoint logits/context and both real attention maps.\n# Before it serializes results, exercise the fresh-text path using the actual\n# Florence/CLIP output produced by a preceding independent process.\n_FRESH_SEGMENTATION_CHECK = r\'\'\'\nfresh_report=net.io_utils.read_json(root/\'example/fresh_caption_check.json\')\nif not all(fresh_report.get(k) for k in (\'forced_fresh_generation\',\'florence_clip_reload\',\'original_clip_embedding_parity\')):\n    raise ValueError(\'Fresh caption engine has not been independently verified\')\nfresh=torch.load(root/\'example/fresh_caption.pt\',map_location=\'cpu\',weights_only=True)\nclass VerifiedFreshCaption:\n    def caption(self,source):\n        rgb=net.io_utils.read_rgb(source)\n        digest=__import__(\'hashlib\').sha256(np.asarray(rgb).tobytes()+str(rgb.size).encode()).hexdigest()\n        if digest!=fresh[\'pixel_sha256\']:\n            raise ValueError(\'Fresh caption verification image changed\')\n        return fresh[\'row\'],fresh[\'vector\'],fresh[\'valid\']\ncached_index=predictor.manifest.get(\'caption_pixel_index\',{})\npredictor.manifest[\'caption_pixel_index\']={}\npredictor.caption_engine=VerifiedFreshCaption()\nnew_result=predictor.predict(image,return_attention=False)\npredictor.manifest[\'caption_pixel_index\']=cached_index\nif new_result[\'evidence\'][\'query_caption\'][\'status\']!=\'fresh\' or not torch.isfinite(new_result[\'logits\']).all():\n    raise ValueError(\'Fresh full_both inference path failed\')\nnet.io_utils.write_json(root/\'example/fresh_segmentation_check.json\',{\n    \'fresh_caption_used\':True,\'selected_context_id\':new_result[\'evidence\'][\'candidate\'][\'image_id\'],\n    \'query_caption\':new_result[\'evidence\'][\'query_caption\'],\'fusion\':new_result[\'evidence\'][\'fusion\']})\n\'\'\'\n\nCOLD_SOURCE = common.COLD_SOURCE.replace(\'torch.save(explained.pop("logits")\',\n    _FRESH_SEGMENTATION_CHECK + \'\\ntorch.save(explained.pop("logits")\').replace(\n    \'"device":device, "benchmark_metric":False\',\n    \'"device":device, "benchmark_metric":False, "fresh_caption_engine":True, "fresh_full_both_inference":True, "original_clip_embedding_parity":True\')\n\n\ndef verify_and_publish(ctx, *, device=None, caption_device=None):\n    if ctx["factory"] is not None:\n        # Explicit tiny model fixtures cannot become ready/deployable.\n        return common.verify_and_publish(ctx, device=device)\n    core, output, stage = ctx["caption_core"], ctx["output"], ctx["stage"]\n    script = output / "verify_fresh_caption.py"\n    script.write_text(FRESH_SOURCE, encoding="utf-8")\n    # Only copied wheels can repair this isolated child. Never change kernel packages.\n    old_path = os.environ.pop("PYTHONPATH", None)\n    try:\n        env = core.caption_runtime_environment("legacy", stage / "offline_wheels", output / "caption_verify_runtime")\n    finally:\n        if old_path is not None:\n            os.environ["PYTHONPATH"] = old_path\n    env.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")\n    log_path = output / "fresh_caption_reload.log"\n    print(f"Forced fresh-caption/CLIP check on {caption_device or device or ctx[\'device\']}; log: {log_path}", flush=True)\n    with log_path.open("w", encoding="utf-8") as log:\n        result = subprocess.run([sys.executable, str(script), str(stage), str(caption_device or device or ctx["device"])],\n            cwd=stage, env=env, stdout=log, stderr=subprocess.STDOUT, timeout=1800)\n    if result.returncode:\n        raise RuntimeError("Fresh caption verification failed; nothing is marked ready.\\n" + log_path.read_text("utf-8")[-6000:])\n    print(log_path.read_text("utf-8")[-1600:])\n    # The next child checks every bound file, including the actual fresh-caption\n    # output and its CLIP parity report. Rebind hashes while status stays staged.\n    common.stage_manifest(ctx)\n    return common.verify_and_publish(ctx, device=device)\n\n\ndef package_downloads(ctx, *, part_mib=256):\n    manifest = common.read_json(ctx["stage"] / "manifest.json")\n    if not all(manifest.get("verification", {}).get(k) for k in\n            ("fresh_caption_engine", "fresh_full_both_inference", "original_clip_embedding_parity")):\n        raise ValueError("Complete M-TACS web export requires verified real Florence/CLIP fresh-image inference")\n    return common.package_downloads(ctx, part_mib=part_mib)\n'}
FALLBACK_SOURCES = {'tacs_visual_web.py': '"""Portable visual TACS model and inference contract for the SUN397 web bundle.\n\nNo captions, CLIP, label prior or external service is used. The same backbone\nencodes selector images and the concatenated query/context patch sequence.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib\nimport io\nimport json\nfrom pathlib import Path, PurePosixPath\nimport sys\n\nimport numpy as np\nfrom PIL import Image, ImageOps\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\nSCHEMA = "sun397_tacs_visual_web_v1"\nATTENTION_METHOD = "last_block_cls_to_patch_attention_mean_heads"\n\n\ndef file_sha256(path):\n    with Path(path).open("rb") as handle:\n        return hashlib.file_digest(handle, "sha256").hexdigest() if hasattr(hashlib, "file_digest") else _hash_stream(handle)\n\n\ndef _hash_stream(handle):\n    digest = hashlib.sha256()\n    for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):\n        digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef bundle_path(root, relative):\n    relative = str(relative)\n    pure = PurePosixPath(relative)\n    if pure.is_absolute() or "\\\\" in relative or ":" in relative or ".." in pure.parts:\n        raise ValueError(f"Unsafe bundle-relative path: {relative}")\n    root = Path(root).resolve()\n    target = root.joinpath(*pure.parts).resolve()\n    if not target.is_relative_to(root):\n        raise ValueError(f"Bundle path escapes root: {relative}")\n    return target\n\n\ndef read_rgb(source):\n    if isinstance(source, Image.Image):\n        return ImageOps.exif_transpose(source).convert("RGB")\n    if isinstance(source, bytes):\n        source = io.BytesIO(source)\n    with Image.open(source) as image:\n        return ImageOps.exif_transpose(image).convert("RGB")\n\n\ndef preprocess(image, config):\n    """The exported evaluation transform, including EXIF and resize semantics."""\n    image = read_rgb(image)\n    size = int(config["image_size"])\n    image = image.resize((size, size), Image.Resampling.BICUBIC)\n    pixels = torch.from_numpy(np.array(image, dtype=np.float32).copy()).permute(2, 0, 1) / 255.0\n    mean = torch.tensor(config["mean"], dtype=torch.float32)[:, None, None]\n    std = torch.tensor(config["std"], dtype=torch.float32)[:, None, None]\n    return (pixels - mean) / std\n\n\ndef concatenate_rope(rope):\n    """Token order is [query patches, context patches], not interleaved."""\n    if rope is None:\n        return None\n    if not isinstance(rope, (tuple, list)) or len(rope) != 2:\n        raise TypeError("Expected DINOv3 RoPE (sin, cos); refusing a positional fallback")\n    return tuple(torch.cat((part, part), dim=-2) for part in rope)\n\n\ndef mask_self(scores, excluded_positions=None):\n    if scores.ndim != 2 or scores.shape[1] < 2:\n        raise ValueError("TACS requires at least two candidates")\n    if excluded_positions is not None:\n        positions = torch.as_tensor(excluded_positions, dtype=torch.long, device=scores.device)\n        if positions.shape != (scores.shape[0],) or (positions >= scores.shape[1]).any():\n            raise ValueError("Invalid query-to-candidate mapping")\n        rows = torch.arange(len(positions), device=scores.device)\n        valid = positions >= 0\n        scores = scores.clone()\n        scores[rows[valid], positions[valid]] = -torch.inf\n    if not torch.isfinite(scores).any(dim=1).all():\n        raise ValueError("No eligible candidate for a query")\n    return scores\n\n\nclass _StraightThroughImage(torch.autograd.Function):\n    """Exact derivative of hard Gumbel weights @ fixed pixels, with CPU banking."""\n\n    @staticmethod\n    def forward(ctx, weights, images, chunk_size):\n        if images.ndim != 4 or weights.shape[1] != len(images):\n            raise ValueError("Image bank/selection shape mismatch")\n        hard = F.one_hot(weights.argmax(1), len(images)).to(weights.dtype)\n        if not torch.equal(weights.detach(), hard):\n            raise ValueError("Straight-through image forward requires hard one-hot values")\n        ctx.save_for_backward(images)\n        ctx.chunk_size = int(chunk_size)\n        ctx.weight_device, ctx.weight_dtype = weights.device, weights.dtype\n        indices = weights.argmax(1).to(images.device)\n        return images.index_select(0, indices).to(weights.device, dtype=torch.float32)\n\n    @staticmethod\n    def backward(ctx, grad_images):\n        (bank,) = ctx.saved_tensors\n        grad = grad_images.float().flatten(1)\n        chunks = []\n        # Float32 avoids an fp16 bank matmul silently changing selector gradients.\n        with torch.autocast(device_type=grad.device.type, enabled=False):\n            for start in range(0, len(bank), ctx.chunk_size):\n                block = bank[start:start + ctx.chunk_size].to(grad.device, dtype=torch.float32)\n                chunks.append(grad @ block.flatten(1).T)\n        return torch.cat(chunks, dim=1).to(ctx.weight_device, ctx.weight_dtype), None, None\n\n\ndef straight_through_images(weights, bank, chunk_size=32):\n    return _StraightThroughImage.apply(weights, bank, chunk_size)\n\n\nclass VisualTACS(nn.Module):\n    def __init__(self, backbone, config):\n        super().__init__()\n        self.backbone = backbone\n        self.config = dict(config)\n        dim = int(getattr(backbone, "embed_dim", config["feature_dim"]))\n        if dim != int(config["feature_dim"]):\n            raise ValueError("Backbone feature dimension does not match run configuration")\n        self.head = nn.Sequential(nn.Dropout(float(config.get("head_dropout", 0.1))), nn.Linear(dim, int(config["num_classes"])))\n        self.gradient_checkpointing = bool(config.get("gradient_checkpointing", True))\n\n    @property\n    def prefix_count(self):\n        return 1 + int(getattr(self.backbone, "n_storage_tokens", 0))\n\n    def _tokens(self, images):\n        prepared = self.backbone.prepare_tokens_with_masks(images, None)\n        if not isinstance(prepared, tuple) or len(prepared) != 2:\n            raise TypeError("This runtime requires DINOv3 prepare_tokens_with_masks -> (tokens, grid)")\n        tokens, grid = prepared\n        height, width = map(int, grid)\n        if tokens.shape[1] != self.prefix_count + height * width:\n            raise ValueError("Unexpected prefix/patch token count")\n        rope = self.backbone.rope_embed(H=height, W=width) if self.backbone.rope_embed is not None and not self.training else None\n        return tokens, rope, (height, width)\n\n    @staticmethod\n    def _cls_attention(block, tokens, rope):\n        attention = block.attn\n        qkv = attention.qkv(block.norm1(tokens))\n        batch, count, triple_dim = qkv.shape\n        dim = triple_dim // 3\n        heads = int(attention.num_heads)\n        q, k, _ = qkv.reshape(batch, count, 3, heads, dim // heads).unbind(2)\n        q, k = q.transpose(1, 2), k.transpose(1, 2)\n        if rope is not None:\n            q, k = attention.apply_rope(q, k, rope)\n        weights = ((q[:, :, :1].float() @ k.float().transpose(-2, -1)) * (dim // heads) ** -0.5).softmax(dim=-1)\n        return weights.mean(dim=1)[:, 0]\n\n    def _encode(self, tokens, rope, return_attention=False, grid=None, paired=False):\n        if return_attention and self.training:\n            raise RuntimeError("Attention export is evaluation-only")\n        cls_attention = None\n        for index, block in enumerate(self.backbone.blocks):\n            block_rope = rope\n            if self.training and self.backbone.rope_embed is not None:\n                # Native DINOv3 resamples positional augmentation per block.\n                # Pass that sample into checkpoint so backward reuses it exactly.\n                block_rope = self.backbone.rope_embed(H=grid[0], W=grid[1])\n                if paired:\n                    block_rope = concatenate_rope(block_rope)\n            if return_attention and index == len(self.backbone.blocks) - 1:\n                cls_attention = self._cls_attention(block, tokens, block_rope)\n            if self.training and self.gradient_checkpointing and torch.is_grad_enabled():\n                tokens = checkpoint(block, tokens, block_rope, use_reentrant=False, preserve_rng_state=True)\n            else:\n                tokens = block(tokens, block_rope)\n        if getattr(self.backbone, "untie_cls_and_patch_norms", False):\n            cls = self.backbone.cls_norm(tokens[:, :self.prefix_count])[:, 0]\n        else:\n            cls = self.backbone.norm(tokens)[:, 0]\n        return cls, cls_attention\n\n    def encode_selector(self, images, return_attention=False):\n        tokens, rope, grid = self._tokens(images)\n        features, attention = self._encode(tokens, rope, return_attention, grid=grid)\n        if return_attention:\n            return features, attention[:, self.prefix_count:].reshape(-1, *grid)\n        return features\n\n    def score(self, query_images, candidate_features, excluded_positions=None):\n        query = self.encode_selector(query_images).float()\n        bank = candidate_features.to(query.device, dtype=torch.float32)\n        # Eq. 1 utility: shared CLS dot product. No class labels enter scoring.\n        with torch.autocast(device_type=query.device.type, enabled=False):\n            scores = query @ bank.T\n        return mask_self(scores, excluded_positions), query\n\n    def forward(self, query_images, context_images=None, return_attention=False):\n        tokens, rope, grid = self._tokens(query_images)\n        patch_count = grid[0] * grid[1]\n        if context_images is not None:\n            context_tokens, _, context_grid = self._tokens(context_images)\n            if grid != context_grid:\n                raise ValueError("Query and context must have the same patch grid")\n            tokens = torch.cat((tokens, context_tokens[:, self.prefix_count:]), dim=1)\n            rope = concatenate_rope(rope)\n        features, attention = self._encode(tokens, rope, return_attention, grid=grid, paired=context_images is not None)\n        logits = self.head(features)\n        if not return_attention:\n            return logits\n        result = {"query_classifier": attention[:, self.prefix_count:self.prefix_count + patch_count].reshape(-1, *grid)}\n        if context_images is not None:\n            result["context_classifier"] = attention[:, self.prefix_count + patch_count:].reshape(-1, *grid)\n        return logits, result\n\n\ndef build_model(repo_dir, config, weights_path=None):\n    if config["backbone"] != "dinov3_vitl16" or int(config["feature_dim"]) != 1024:\n        raise ValueError("Production bundle requires DINOv3 ViT-L/16, 1024 dimensions; no S/B substitution")\n    repo = Path(repo_dir).resolve()\n    if not (repo / "dinov3/hub/backbones.py").is_file():\n        raise FileNotFoundError(f"DINOv3 source/dinov3/hub/backbones.py missing: {repo}")\n    kwargs = {"pretrained": weights_path is not None}\n    if weights_path is not None:\n        weights = Path(weights_path).resolve()\n        if not weights.is_file():\n            raise FileNotFoundError(weights)\n        if not weights.name.lower().endswith("lvd1689m-8aa4cbdd.pth"):\n            raise ValueError("Use the official ViT-L/16 LVD1689M 8aa4cbdd .pth with its original filename; SAT/plus formats are not supported here")\n        kwargs["weights"] = str(weights)\n    # hubconf also imports unrelated detector/segmentor dependencies. Use the\n    # exact official backbone factory without executing those entry points.\n    package = sys.modules.get("dinov3")\n    if package is not None and not Path(package.__file__).resolve().is_relative_to(repo):\n        raise RuntimeError("Another DINOv3 source is already imported. Restart the kernel before loading this bundle.")\n    sys.path.insert(0, str(repo))\n    backbones = importlib.import_module("dinov3.hub.backbones")\n    if not Path(backbones.__file__).resolve().is_relative_to(repo):\n        raise RuntimeError("DINOv3 backbone code did not load from the requested local repository")\n    backbone = backbones.dinov3_vitl16(**kwargs)\n    if len(backbone.blocks) != 24:\n        raise ValueError("Expected 24 transformer blocks for ViT-L/16")\n    return VisualTACS(backbone, config)\n\n\ndef normalize_map(raw):\n    raw = np.asarray(raw, dtype=np.float32)\n    if raw.ndim != 2 or not np.isfinite(raw).all() or raw.min() < 0:\n        raise ValueError("Invalid attention map")\n    span = float(raw.max() - raw.min())\n    return (raw - raw.min()) / max(span, 1e-12), span > 1e-12\n\n\ndef attention_overlay(image, raw, alpha=0.45):\n    image = read_rgb(image)\n    normalized, informative = normalize_map(raw)\n    heat = Image.fromarray(np.uint8(np.round(normalized * 255))).resize(image.size, Image.Resampling.BILINEAR)\n    values = np.asarray(heat, dtype=np.float32) / 255.0\n    colors = np.stack((values, 0.25 * (1 - values), 1 - values), axis=-1)\n    pixels = np.asarray(image, dtype=np.float32) / 255.0\n    mixed = (1 - alpha) * pixels + alpha * colors\n    # Flat attention is preserved and flagged, never replaced by invented structure.\n    return Image.fromarray(np.uint8(np.clip(mixed * 255, 0, 255))), informative\n\n\ndef png_data_url(image):\n    import base64\n    handle = io.BytesIO()\n    image.save(handle, format="PNG")\n    return "data:image/png;base64," + base64.b64encode(handle.getvalue()).decode("ascii")\n\n\ndef verify_bundle(root):\n    root = Path(root).resolve()\n    manifest = read_json(root / "manifest.json")\n    if manifest.get("schema") != SCHEMA or manifest.get("status") != "ready":\n        raise ValueError("Bundle is incomplete or uses another schema")\n    files = manifest.get("files", {})\n    required = {"model.pt", "config.json", "class_names.json", "candidate_pool.json", "candidate_features.pt",\n                "runtime/tacs_visual_web.py", "vendor/dinov3/dinov3/hub/backbones.py"}\n    if not required.issubset(files):\n        raise ValueError("Manifest omits a required inference artifact")\n    for name, info in files.items():\n        path = bundle_path(root, name)\n        if not path.is_file() or path.stat().st_size != int(info["bytes"]) or file_sha256(path) != info["sha256"]:\n            raise ValueError(f"Bundle checksum mismatch: {name}")\n    return manifest\n\n\nclass BundlePredictor:\n    """Load a trusted, exported bundle. No training Inputs or network are needed."""\n    def __init__(self, root, device="cpu", model_factory=build_model):\n        self.root = Path(root).resolve()\n        self.manifest = verify_bundle(self.root)\n        self.config = read_json(self.root / "config.json")\n        self.class_names = read_json(self.root / "class_names.json")\n        self.candidates = read_json(self.root / "candidate_pool.json")\n        self.device = torch.device(device)\n        if len(self.class_names) != self.config["num_classes"] or len(set(self.class_names)) != len(self.class_names):\n            raise ValueError("Class order/count is invalid")\n        if len(self.candidates) != self.config["candidate_count"]:\n            raise ValueError("Candidate count mismatch")\n        ids = [row["image_id"] for row in self.candidates]\n        if len(set(ids)) != len(ids):\n            raise ValueError("Duplicate candidate IDs")\n        for position, row in enumerate(self.candidates):\n            if row["position"] != position or row["file"] not in self.manifest["files"]:\n                raise ValueError("Candidate bank order or image manifest is invalid")\n            if row["sha256"] != self.manifest["files"][row["file"]]["sha256"]:\n                raise ValueError("Candidate content hash does not match its image")\n            if self.class_names[row["class_id"]] != row["class_name"]:\n                raise ValueError("Candidate class mapping mismatch")\n            bundle_path(self.root, row["file"])\n        checkpoint_state = torch.load(self.root / "model.pt", map_location="cpu", weights_only=True)\n        if checkpoint_state["config"] != self.config or checkpoint_state["class_names"] != self.class_names:\n            raise ValueError("Model/config/class mapping mismatch")\n        bank = torch.load(self.root / "candidate_features.pt", map_location="cpu", weights_only=True)\n        if bank["image_ids"] != ids or bank["checkpoint_sha256"] != file_sha256(self.root / "model.pt"):\n            raise ValueError("Candidate features belong to another checkpoint/order")\n        if tuple(bank["features"].shape) != (len(ids), self.config["feature_dim"]) or not torch.isfinite(bank["features"]).all():\n            raise ValueError("Candidate features are invalid")\n        self.model = model_factory(self.root / "vendor/dinov3", self.config).to(self.device)\n        self.model.load_state_dict(checkpoint_state["model_state_dict"], strict=True)\n        self.model.eval()\n        self.bank = bank["features"].float().to(self.device)\n        self.last_logits = None\n\n    def predict(self, source, top_k=5, explain=True, image_id=None):\n        image = read_rgb(source)\n        query = preprocess(image, self.config).unsqueeze(0).to(self.device)\n        excluded = torch.zeros(len(self.candidates), dtype=torch.bool, device=self.device)\n        if isinstance(source, (str, Path)):\n            content_hash = file_sha256(source)\n        elif isinstance(source, bytes):\n            content_hash = hashlib.sha256(source).hexdigest()\n        else:\n            content_hash = None\n        for i, row in enumerate(self.candidates):\n            if row["image_id"] == image_id or (content_hash is not None and row["sha256"] == content_hash):\n                excluded[i] = True\n        with torch.no_grad():\n            scores, _ = self.model.score(query, self.bank)\n            scores = scores.masked_fill(excluded.unsqueeze(0), -torch.inf)\n            if not torch.isfinite(scores).any():\n                raise ValueError("No eligible candidate after self-exclusion")\n            position = int(scores.argmax(1).item())\n            row = self.candidates[position]\n            candidate = read_rgb(bundle_path(self.root, row["file"]))\n            context = preprocess(candidate, self.config).unsqueeze(0).to(self.device)\n            if explain:\n                logits, maps = self.model(query, context, return_attention=True)\n                _, selector_map = self.model.encode_selector(context, return_attention=True)\n            else:\n                logits = self.model(query, context)\n            self.last_logits = logits.detach().cpu()\n            probabilities = logits.float().softmax(1)[0]\n            values, indices = probabilities.topk(min(max(1, int(top_k)), len(self.class_names)))\n        result = {\n            "model": "tacs", "dataset": self.config["dataset"], "bundle_schema": SCHEMA,\n            "prediction": {"class_id": int(indices[0]), "label": self.class_names[int(indices[0])], "confidence": float(values[0])},\n            "top_k": [{"class_id": int(i), "label": self.class_names[int(i)], "confidence": float(v)} for v, i in zip(values, indices)],\n            "evidence": {\n                "query_caption": {"text": None, "status": "not_used"},\n                "candidate": {"image_id": row["image_id"], "position": position, "image_url": png_data_url(candidate), "caption": None,\n                              "selector_score": float(scores[0, position]), "status": "ready"},\n                "attention": {},\n            },\n        }\n        if explain:\n            for name, raw, source_image in [("query_classifier", maps["query_classifier"][0], image),\n                                            ("candidate_selector", selector_map[0], candidate)]:\n                array = raw.float().cpu().numpy()\n                overlay, informative = attention_overlay(source_image, array)\n                result["evidence"]["attention"][name] = {\n                    "image_url": png_data_url(overlay), "status": "ready" if informative else "flat_attention",\n                    "method": ATTENTION_METHOD, "raw_grid": array.tolist(), "grid_shape": list(array.shape),\n                    "normalization": "per_image_min_max_for_display_only", "class_conditional": False,\n                }\n        return result\n\n\ndef main():\n    import argparse\n    parser = argparse.ArgumentParser(description="Offline visual TACS SUN397 bundle inference")\n    parser.add_argument("--bundle", type=Path, required=True)\n    parser.add_argument("--image", type=Path, required=True)\n    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\n    parser.add_argument("--output", type=Path, default=Path("prediction.json"))\n    parser.add_argument("--no-attention", action="store_true")\n    args = parser.parse_args()\n    predictor = BundlePredictor(args.bundle, args.device)\n    write_json(args.output, predictor.predict(args.image, explain=not args.no_attention))\n    print(f"Saved prediction and evidence to {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n', 'isic_segmentation_web.py': '"""Frozen DINOv3 + per-block gated context attention for ISIC deployment.\n\nOne frozen backbone is shared by selector feature extraction and segmentation.\nOnly utility projections, inserted attention, DPT and optional text fusion learn.\nMasks and diagnostic labels are never inference or context-selection inputs.\n"""\nfrom __future__ import annotations\n\nimport importlib\nimport json\nimport math\nfrom pathlib import Path\nimport sys\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\n\nimport tacs_visual_web as io_utils\n\n\nSCHEMA = "isic2017_segmentation_web_v1"\nBACKBONES = {"dinov3_vitl16": (1024, 24, 16, "8aa4cbdd"),\n             "dinov3_vits16": (384, 12, 6, "08c60483")}\nATTENTION_METHOD = "last_block_cls_to_patch_mean_heads"\n\n\ndef build_backbone(repo, config, weights=None):\n    name = config["backbone"]\n    if name not in BACKBONES or config["feature_dim"] != BACKBONES[name][0]:\n        raise ValueError("Backbone/feature dimension mismatch; no silent model substitution")\n    repo = Path(repo).resolve()\n    if not (repo / "dinov3/hub/backbones.py").is_file():\n        raise FileNotFoundError(f"Missing official DINOv3 source: {repo}")\n    package = sys.modules.get("dinov3")\n    if package is not None and not Path(package.__file__).resolve().is_relative_to(repo):\n        raise RuntimeError("A different DINOv3 repository is imported; restart the kernel")\n    sys.path.insert(0, str(repo))\n    module = importlib.import_module("dinov3.hub.backbones")\n    if not Path(module.__file__).resolve().is_relative_to(repo):\n        raise RuntimeError("DINOv3 source did not resolve inside the requested repository")\n    kwargs = {"pretrained": weights is not None}\n    if weights is not None:\n        weights = Path(weights).resolve()\n        suffix = f"lvd1689m-{BACKBONES[name][3]}.pth"\n        if not weights.is_file() or not weights.name.endswith(suffix):\n            raise ValueError(f"Use the original official {name} LVD1689M checkpoint ({suffix})")\n        kwargs["weights"] = str(weights)\n    backbone = getattr(module, name)(**kwargs)\n    if backbone.embed_dim != config["feature_dim"] or len(backbone.blocks) != BACKBONES[name][1]:\n        raise ValueError("Loaded backbone has the wrong width or depth")\n    return backbone\n\n\ndef build_model(repo, config, weights=None):\n    return ISICSegmentor(build_backbone(repo, config, weights), config)\n\n\nclass _HardBank(torch.autograd.Function):\n    """Exact ST derivative w.r.t. selection weights over a fixed CPU token bank."""\n    @staticmethod\n    def forward(ctx, weights, bank, chunk):\n        if weights.ndim != 2 or weights.shape[1] != len(bank):\n            raise ValueError("Selection/token bank shape mismatch")\n        one_hot = F.one_hot(weights.argmax(1), len(bank)).to(weights.dtype)\n        if not torch.equal(weights.detach(), one_hot):\n            raise ValueError("Hard forward requires one-hot ST weights")\n        ctx.save_for_backward(bank)\n        ctx.chunk = int(chunk)\n        ctx.weight_dtype = weights.dtype\n        return bank.index_select(0, weights.argmax(1).to(bank.device)).to(weights.device, dtype=torch.float32)\n\n    @staticmethod\n    def backward(ctx, gradient):\n        (bank,) = ctx.saved_tensors\n        pieces = []\n        with torch.autocast(device_type=gradient.device.type, enabled=False):\n            for offset in range(0, len(bank), ctx.chunk):\n                block = bank[offset:offset + ctx.chunk].to(gradient.device, dtype=torch.float32)\n                pieces.append(gradient.float().flatten(1) @ block.flatten(1).T)\n        return torch.cat(pieces, 1).to(ctx.weight_dtype), None, None\n\n\ndef select_bank(weights, bank, chunk=8):\n    return _HardBank.apply(weights, bank, chunk)\n\n\nclass UtilityHead(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        # Frozen CLS features alone cannot learn utility. These projections learn\n        # from BOTH task gradients and policy gradients, sharing the DINO encoder.\n        self.query = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(), nn.Linear(dim, dim))\n        self.candidate = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(), nn.Linear(dim, dim))\n        self.log_scale = nn.Parameter(torch.tensor(1.0))\n\n    def forward(self, query, candidates):\n        with torch.autocast(device_type=query.device.type, enabled=False):\n            q = F.normalize(self.query(query.float()), dim=-1)\n            c = F.normalize(self.candidate(candidates.float()), dim=-1)\n            return (q @ c.T) * self.log_scale.clamp(0.01, 100)\n\n\nclass GatedContextAttention(nn.Module):\n    def __init__(self, dim, heads):\n        super().__init__()\n        self.query_norm = nn.LayerNorm(dim)\n        self.context_norm = nn.LayerNorm(dim)\n        self.attention = nn.MultiheadAttention(dim, heads, dropout=0.0, batch_first=True)\n        self.gate_logit = nn.Parameter(torch.tensor(-4.0))\n\n    def forward(self, query, context):\n        q, c = self.query_norm(query), self.context_norm(context)\n        update, _ = self.attention(q, c, c, need_weights=False)\n        return query + torch.sigmoid(self.gate_logit) * update\n\n\nclass ResidualConv(nn.Module):\n    def __init__(self, channels):\n        super().__init__()\n        self.layers = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1),\n            nn.GroupNorm(1, channels), nn.GELU(), nn.Conv2d(channels, channels, 3, padding=1),\n            nn.GroupNorm(1, channels), nn.GELU())\n\n    def forward(self, x):\n        return x + self.layers(x)\n\n\nclass DPTLite(nn.Module):\n    """Four-depth reassembly at 4x/2x/1x/0.5x and top-down residual fusion."""\n    def __init__(self, dim, channels):\n        super().__init__()\n        self.project = nn.ModuleList([nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, channels)) for _ in range(4)])\n        self.resize = nn.ModuleList([nn.ConvTranspose2d(channels, channels, 4, stride=4),\n            nn.ConvTranspose2d(channels, channels, 2, stride=2), nn.Identity(),\n            nn.Conv2d(channels, channels, 3, stride=2, padding=1)])\n        self.refine = nn.ModuleList([ResidualConv(channels) for _ in range(4)])\n        self.head = nn.Sequential(nn.Conv2d(channels, channels // 2, 3, padding=1), nn.GELU(),\n                                  nn.Conv2d(channels // 2, 1, 1))\n\n    def forward(self, stages, grid, size):\n        if len(stages) != 4:\n            raise ValueError("DPT requires four distinct transformer depths")\n        maps = [resize(proj(tokens).transpose(1, 2).reshape(tokens.shape[0], -1, *grid))\n                for tokens, proj, resize in zip(stages, self.project, self.resize)]\n        x = self.refine[3](maps[3])\n        for index in (2, 1, 0):\n            x = F.interpolate(x, size=maps[index].shape[-2:], mode="bilinear", align_corners=False)\n            x = self.refine[index](x + maps[index])\n        return F.interpolate(self.head(x), size=(size, size), mode="bilinear", align_corners=False)\n\n\nclass TextFusion(nn.Module):\n    def __init__(self, text_dim, dim):\n        super().__init__()\n        self.projection = nn.Sequential(nn.LayerNorm(text_dim), nn.Linear(text_dim, dim), nn.GELU())\n        self.gate = nn.Linear(dim * 2, 1)\n        nn.init.zeros_(self.gate.weight)\n        nn.init.constant_(self.gate.bias, -4.0)\n\n    def forward(self, patches, text, valid):\n        projected = self.projection(text.float()).to(patches.dtype)\n        gate = torch.sigmoid(self.gate(torch.cat((patches.mean(1).float(), projected.float()), -1)))\n        gate = gate * valid.float().reshape(-1, 1)\n        return patches + gate[:, :, None] * projected[:, None, :], gate[:, 0]\n\n\nclass ISICSegmentor(nn.Module):\n    def __init__(self, backbone, config):\n        super().__init__()\n        self.backbone = backbone.eval().requires_grad_(False)\n        self.config = dict(config)\n        dim = int(backbone.embed_dim)\n        if dim != int(config["feature_dim"]):\n            raise ValueError("Backbone/config dimension mismatch")\n        self.prefix = 1 + int(backbone.n_storage_tokens)\n        self.grid = (config["image_size"] // 16,) * 2\n        self.stage_indices = [round(len(backbone.blocks) * fraction) - 1 for fraction in (.25, .5, .75, 1)]\n        if len(set(self.stage_indices)) != 4:\n            raise ValueError("Backbone must expose four distinct depths")\n        self.selector = UtilityHead(dim)\n        self.context_blocks = nn.ModuleList([GatedContextAttention(dim, config["cross_heads"])\n                                            for _ in backbone.blocks])\n        self.decoder = DPTLite(dim, config["decoder_channels"])\n        self.multimodal = config["mode"] == "mtacs"\n        if self.multimodal:\n            self.query_text = TextFusion(config["text_dim"], dim)\n            self.candidate_text = TextFusion(config["text_dim"], dim)\n            self.semantic_mix_logit = nn.Parameter(torch.tensor(0.0))\n\n    def train(self, mode=True):\n        super().train(mode)\n        # Frozen weights AND deterministic pretrained operations; allow input\n        # gradients through them after trainable text/context blocks are added.\n        self.backbone.eval()\n        return self\n\n    def rope(self):\n        return self.backbone.rope_embed(H=self.grid[0], W=self.grid[1]) if self.backbone.rope_embed is not None else None\n\n    @torch.no_grad()\n    def prepare(self, images):\n        tokens, grid = self.backbone.prepare_tokens_with_masks(images, None)\n        if tuple(grid) != self.grid or tokens.shape[1] != self.prefix + math.prod(self.grid):\n            raise ValueError("Unexpected DINOv3 grid/prefix count")\n        return tokens.float()\n\n    def _cls_map(self, block, tokens, rope):\n        # Same qkv, key-bias mask and RoPE as the native DINOv3 attention.\n        return io_utils.VisualTACS._cls_attention(block, tokens, rope)[:, self.prefix:].reshape(-1, *self.grid)\n\n    def _norm_cls(self, tokens):\n        norm = self.backbone.cls_norm if self.backbone.untie_cls_and_patch_norms else self.backbone.norm\n        return norm(tokens[:, :self.prefix])[:, 0].float()\n\n    @torch.no_grad()\n    def encode_selector_tokens(self, initial, return_attention=False):\n        tokens, rope, attention = initial, self.rope(), None\n        for index, block in enumerate(self.backbone.blocks):\n            if return_attention and index == len(self.backbone.blocks) - 1:\n                attention = self._cls_map(block, tokens, rope)\n            tokens = block(tokens, rope)\n        return self._norm_cls(tokens), attention\n\n    @torch.no_grad()\n    def encode_selector(self, images, return_attention=False):\n        return self.encode_selector_tokens(self.prepare(images), return_attention)\n\n    def score(self, global_query, global_bank, allowed, q_text=None, q_valid=None, c_text=None, c_valid=None):\n        scores = self.selector(global_query, global_bank.to(global_query.device))\n        if self.multimodal:\n            if any(value is None for value in (q_text, q_valid, c_text, c_valid)):\n                raise ValueError("full_both requires both query and candidate text plus masks")\n            semantic = F.normalize(q_text.float(), dim=1) @ F.normalize(c_text.float(), dim=1).T\n            mask = q_valid.bool()[:, None] & c_valid.bool()[None, :]\n            scores = scores + self.semantic_mix_logit.sigmoid() * semantic * mask\n        if allowed.shape != scores.shape or not allowed.any(1).all():\n            raise ValueError("No eligible candidate or invalid exclusion mask")\n        return scores.masked_fill(~allowed.bool(), -torch.inf)\n\n    def fuse(self, tokens, text, valid, candidate=False):\n        if not self.multimodal:\n            return tokens, tokens.new_zeros(len(tokens))\n        layer = self.candidate_text if candidate else self.query_text\n        patches, gate = layer(tokens[:, self.prefix:], text, valid)\n        return torch.cat((tokens[:, :self.prefix], patches), 1), gate\n\n    def decode(self, query_initial, context_initial=None, q_text=None, q_valid=None,\n               c_text=None, c_valid=None, return_attention=False):\n        if return_attention and self.training:\n            raise ValueError("Attention is evaluation-only")\n        query, q_gate = self.fuse(query_initial, q_text, q_valid)\n        context, c_gate = (self.fuse(context_initial, c_text, c_valid, True) if context_initial is not None\n                           else (None, query.new_zeros(len(query))))\n        rope, stages, query_attention = self.rope(), [], None\n        for index, (block, fusion) in enumerate(zip(self.backbone.blocks, self.context_blocks)):\n            if return_attention and index == len(self.backbone.blocks) - 1:\n                query_attention = self._cls_map(block, query, rope)\n            # Bind modules in defaults: checkpoint\'s backward must NOT capture\n            # the loop\'s final block via a late-bound closure.\n            def step(q, c, block=block, fusion=fusion):\n                q = block(q, rope)\n                if c is None:\n                    return q\n                c = block(c, rope)\n                patch = fusion(q[:, self.prefix:], c[:, self.prefix:])\n                return torch.cat((q[:, :self.prefix], patch), 1), c\n            if self.training and self.config.get("gradient_checkpointing", True) and torch.is_grad_enabled():\n                result = checkpoint(step, query, context, use_reentrant=False, preserve_rng_state=True)\n            else:\n                result = step(query, context)\n            if context is None:\n                query = result\n            else:\n                query, context = result\n            if index in self.stage_indices:\n                stages.append(self.backbone.norm(query[:, self.prefix:]))\n        logits = self.decoder(stages, self.grid, self.config["image_size"])\n        details = {"q_gate": q_gate, "c_gate": c_gate}\n        if return_attention:\n            details["query_segmentor"] = query_attention\n        return logits, details\n\n    def trainable_state(self):\n        return {name: value.detach().cpu().clone() for name, value in self.state_dict().items()\n                if not name.startswith("backbone.")}\n\n    def restore_trainable(self, state):\n        expected = {name for name in self.state_dict() if not name.startswith("backbone.")}\n        if set(state) != expected:\n            raise ValueError("Trainable checkpoint keys differ from the model architecture")\n        result = self.load_state_dict(state, strict=False)\n        if result.unexpected_keys or any(not key.startswith("backbone.") for key in result.missing_keys):\n            raise ValueError("Incomplete trainable state load")\n\n\ndef segmentation_loss(logits, masks):\n    bce = F.binary_cross_entropy_with_logits(logits.float(), masks.float(), reduction="none").mean((1, 2, 3))\n    probabilities = logits.float().sigmoid()\n    intersection = (probabilities * masks).sum((1, 2, 3))\n    dice = (2 * intersection + 1e-6) / (probabilities.sum((1, 2, 3)) + masks.sum((1, 2, 3)) + 1e-6)\n    return bce + 1 - dice\n\n\ndef binary_metrics(logits, masks, threshold=.5):\n    prediction, truth = logits.float().sigmoid() >= threshold, masks.bool()\n    intersection = (prediction & truth).sum((1, 2, 3)).float()\n    union = (prediction | truth).sum((1, 2, 3)).float()\n    total = prediction.sum((1, 2, 3)).float() + truth.sum((1, 2, 3)).float()\n    return (2 * intersection + 1e-6) / (total + 1e-6), (intersection + 1e-6) / (union + 1e-6)\n\n\ndef verify_bundle(root, *, allow_staged=False, allow_fixture=False):\n    root = Path(root).resolve()\n    manifest = io_utils.read_json(root / "manifest.json")\n    statuses = {"ready"} | ({"staged"} if allow_staged else set()) | ({"fixture_unverified"} if allow_fixture else set())\n    if manifest.get("status") == "fixture_unverified" and not allow_fixture:\n        raise ValueError("Test fixture export cannot be used as a verified production model")\n    if manifest.get("schema") != SCHEMA or manifest.get("status") not in statuses:\n        raise ValueError("Wrong/incomplete ISIC deployment manifest")\n    for name, digest in manifest["files"].items():\n        if io_utils.file_sha256(io_utils.bundle_path(root, name)) != digest:\n            raise ValueError(f"Bundle integrity failure: {name}")\n    config = io_utils.read_json(root / "config.json")\n    if manifest["mode"] != config["mode"] or config["context_top_k"] != 1:\n        raise ValueError("Bundle mode/K does not match configuration")\n    return manifest, config\n\n\nclass Predictor:\n    def __init__(self, root, device="cpu", *, factory=None, caption_engine=None, _allow_staged=False):\n        self.root = Path(root).resolve()\n        self.manifest, self.config = verify_bundle(self.root, allow_staged=_allow_staged, allow_fixture=factory is not None)\n        self.device = torch.device(device)\n        repo = io_utils.bundle_path(self.root, "vendor/dinov3")\n        weights = io_utils.bundle_path(self.root, self.manifest["backbone_file"])\n        self.model = (factory(self.config) if factory else build_model(repo, self.config, weights)).to(self.device).eval()\n        state = {}\n        for name in ("models/selector.pt", "models/segmentor.pt"):\n            piece = torch.load(self.root / name, map_location="cpu", weights_only=True)\n            if set(state) & set(piece):\n                raise ValueError("Duplicate state keys in model components")\n            state.update(piece)\n        self.model.restore_trainable(state)\n        self.bank = torch.load(self.root / "candidate_bank.pt", map_location="cpu", weights_only=True)\n        self.pool = io_utils.read_json(self.root / "candidate_pool.json")\n        self.ground_truth_index = io_utils.read_json(self.root / "ground_truth_index.json")\n        ids = [row["image_id"] for row in self.pool]\n        if self.bank["image_ids"] != ids or len(ids) < 2 or len(set(ids)) != len(ids):\n            raise ValueError("Candidate bank/manifest ordering differs")\n        self.caption_engine = caption_engine\n        self.text = None\n        if self.config["mode"] == "mtacs":\n            self.text = torch.load(self.root / "text_embeddings.pt", map_location="cpu", weights_only=True)\n            self.text_position = {image_id: index for index, image_id in enumerate(self.text["image_ids"])}\n            if any(image_id not in self.text_position for image_id in ids):\n                raise ValueError("Missing candidate captions")\n            positions = [self.text_position[image_id] for image_id in ids]\n            self.c_text = self.text["embeddings"][positions].to(self.device)\n            self.c_valid = self.text["valid"][positions].to(self.device)\n            self.captions = io_utils.read_json(self.root / "caption_lookup.json")\n\n    @torch.no_grad()\n    def predict(self, source, *, return_attention=True):\n        import hashlib\n        original = io_utils.read_rgb(source)\n        pixels = io_utils.preprocess(original, self.config).unsqueeze(0).to(self.device)\n        initial = self.model.prepare(pixels)\n        global_query, _ = self.model.encode_selector_tokens(initial)\n        # Content matches may supply captions/ground truth for display, never a\n        # filename-based guess. Inference itself accepts image pixels only.\n        pixel_hash = hashlib.sha256(np.asarray(original).tobytes() + str(original.size).encode()).hexdigest()\n        annotation = self.ground_truth_index.get(pixel_hash)\n        group = annotation.get("group_id") if annotation else None\n        allowed = torch.tensor([[row["pixel_sha256"] != pixel_hash and\n                                not (group and group == row.get("group_id")) for row in self.pool]], device=self.device)\n        q_text = q_valid = None\n        query_caption = {"text": None, "valid": False, "status": "not_used"}\n        if self.text is not None:\n            by_pixels = self.manifest.get("caption_pixel_index", {}).get(pixel_hash)\n            if by_pixels:\n                index = self.text_position[by_pixels]\n                q_text = self.text["embeddings"][index:index + 1].to(self.device)\n                q_valid = self.text["valid"][index:index + 1].to(self.device)\n                query_caption = {"text": self.captions[by_pixels]["final_caption"], "valid": bool(q_valid[0]),\n                                 "status": "cached", "source": "verified_pixels"}\n            else:\n                if self.caption_engine is None:\n                    raise RuntimeError("A new M-TACS image requires the bundled Florence/CLIP caption engine")\n                row, vector, valid = self.caption_engine.caption(original)\n                q_text, q_valid = vector.reshape(1, -1).to(self.device), valid.reshape(1).to(self.device)\n                query_caption = {"text": row["final_caption"], "valid": bool(q_valid[0]),\n                                 "status": "fresh", "source": "florence_prefix"}\n        scores = self.model.score(global_query, self.bank["features"].to(self.device), allowed,\n                                  q_text, q_valid, getattr(self, "c_text", None), getattr(self, "c_valid", None))\n        position = int(scores.argmax(1))\n        candidate = self.pool[position]\n        context = self.bank["tokens"][position:position + 1].to(self.device, dtype=torch.float32)\n        c_text = self.c_text[position:position + 1] if self.text is not None else None\n        c_valid = self.c_valid[position:position + 1] if self.text is not None else None\n        logits, details = self.model.decode(initial, context, q_text, q_valid, c_text, c_valid,\n                                             return_attention=return_attention)\n        probabilities = F.interpolate(logits.float().sigmoid(), size=(original.height, original.width),\n                                      mode="bilinear", align_corners=False)[0, 0].cpu().numpy()\n        mask = probabilities >= self.config["threshold"]\n        mask_image = Image.fromarray((mask * 255).astype(np.uint8))\n        overlay = np.asarray(original, dtype=np.float32).copy()\n        overlay[mask] = .6 * overlay[mask] + .4 * np.array([0, 220, 130])\n        evidence = {"query_caption": query_caption, "candidate": {\n            "image_id": candidate["image_id"], "position": position,\n            "image_url": io_utils.png_data_url(io_utils.read_rgb(self.root / candidate["image"])),\n            "caption": self.captions[candidate["image_id"]]["final_caption"] if self.text is not None else None,\n            "caption_valid": bool(c_valid[0]) if c_valid is not None else False},\n            "fusion": {"q_gate": float(details["q_gate"][0]), "c_gate": float(details["c_gate"][0])}}\n        if return_attention:\n            candidate_image = io_utils.read_rgb(self.root / candidate["image"])\n            _, candidate_map = self.model.encode_selector(io_utils.preprocess(candidate_image, self.config)[None].to(self.device), True)\n            attention = {}\n            for name, image, values in [("query_segmentor", original, details["query_segmentor"][0]),\n                                        ("candidate_selector", candidate_image, candidate_map[0])]:\n                raw = values.float().cpu().numpy()\n                heatmap, informative = io_utils.attention_overlay(image, raw)\n                attention[name] = {"image_url": io_utils.png_data_url(heatmap), "grid": raw.tolist(),\n                    "grid_shape": list(raw.shape), "method": ATTENTION_METHOD, "informative": informative}\n            evidence["attention"] = attention\n        ground_truth, comparison = {"status": "unavailable", "verified": False}, {"status": "unavailable"}\n        if annotation:\n            # Annotation is accessed only AFTER the prediction. It supplies no\n            # input to the segmentor, utility head, captioner or candidate bank.\n            with Image.open(io_utils.bundle_path(self.root, annotation["mask"])) as source_mask:\n                truth = np.asarray(source_mask.convert("L")) > 0\n            if truth.shape != mask.shape:\n                raise ValueError("Verified image and exported ground-truth mask have different dimensions")\n            ground_truth = {"status": "verified_pixels", "verified": True, "image_id": annotation["image_id"],\n                "split": annotation["split"], "mask_url": io_utils.png_data_url(Image.fromarray(truth.astype(np.uint8) * 255))}\n            overlap = float((truth & mask).sum())\n            comparison = {"status": "available", "resolution": "original_image", "dice":\n                (2 * overlap + 1e-6) / (float(truth.sum() + mask.sum()) + 1e-6), "iou":\n                (overlap + 1e-6) / (float((truth | mask).sum()) + 1e-6)}\n        return {"model": self.config["mode"], "task": "binary_segmentation", "logits": logits.cpu(),\n            "probabilities": probabilities, "prediction": {"mask_url": io_utils.png_data_url(mask_image),\n                "overlay_url": io_utils.png_data_url(Image.fromarray(overlay.clip(0, 255).astype(np.uint8))),\n                "foreground_fraction": float(mask.mean()), "threshold": self.config["threshold"],\n                "original_size": list(original.size)}, "evidence": evidence,\n                "ground_truth": ground_truth, "comparison": comparison}\n\n\ndef main():\n    import argparse\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--bundle", required=True)\n    parser.add_argument("--image", required=True)\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--device", default="cpu")\n    parser.add_argument("--threads", type=int, default=2)\n    parser.add_argument("--captions", action="store_true")\n    parser.add_argument("--verify-staged", action="store_true", help=argparse.SUPPRESS)\n    args = parser.parse_args()\n    torch.set_num_threads(args.threads)\n    captioner = None\n    config = io_utils.read_json(Path(args.bundle) / "config.json")\n    if args.captions and config["mode"] == "mtacs":\n        from isic_caption_web import BundledCaptionEngine\n        captioner = BundledCaptionEngine(args.bundle, args.device)\n    result = Predictor(args.bundle, args.device, caption_engine=captioner, _allow_staged=args.verify_staged).predict(args.image)\n    output = Path(args.output)\n    output.mkdir(parents=True, exist_ok=True)\n    torch.save(result.pop("logits"), output / "logits.pt")\n    np.save(output / "probabilities.npy", result.pop("probabilities"))\n    import base64\n    for name, url in [("mask", result["prediction"]["mask_url"]), ("overlay", result["prediction"]["overlay_url"])]:\n        (output / f"{name}.png").write_bytes(base64.b64decode(url.split(",", 1)[1]))\n    io_utils.write_json(output / "prediction.json", result)\n\n\nif __name__ == "__main__":\n    main()\n', 'train_isic_segmentation_web.py': '"""Train/evaluate/export the same ISIC segmentation model used for web inference."""\nfrom __future__ import annotations\n\nfrom contextlib import nullcontext\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch.utils.data import DataLoader, Dataset\nfrom tqdm.auto import tqdm\n\nimport isic_segmentation_web as net\nfrom isic_segmentation_web import io_utils as io\n\n\ndef default_config(mode="tacs"):\n    return {"schema": net.SCHEMA, "mode": mode, "backbone": "dinov3_vitl16", "feature_dim": 1024,\n        "image_size": 448, "cross_heads": 16, "decoder_channels": 128, "text_dim": 512,\n        "context_top_k": 1, "pool_fraction": .20, "seed": 42, "epochs": 100,\n        "batch_size": 2, "effective_batch_size": 8, "eval_batch_size": 4, "feature_batch_size": 8,\n        "workers": 0, "lr": 1e-4, "weight_decay": 1e-4, "warmup_epochs": 5,\n        "gumbel_tau": .1, "policy_weight": .5, "entropy_weight": 0., "grad_clip": 1.,\n        "gradient_checkpointing": True, "threshold": .5, "pool_token_dtype": "float16",\n        "mean": [.485, .456, .406], "std": [.229, .224, .225], "patience": 0,\n        "strict_official_counts": True}\n\n\ndef validate_config(config):\n    if config["mode"] not in {"tacs", "mtacs"} or config["context_top_k"] != 1:\n        raise ValueError("This experiment supports TACS / M-TACS full_both, K=1")\n    if config["image_size"] < 32 or config["image_size"] % 16:\n        raise ValueError("image_size must be a multiple of 16")\n    if config["feature_dim"] % config["cross_heads"] or config["decoder_channels"] < 8:\n        raise ValueError("Invalid attention heads/decoder width")\n    for name in ("epochs", "batch_size", "effective_batch_size", "eval_batch_size", "feature_batch_size"):\n        if type(config[name]) is not int or config[name] < 1:\n            raise ValueError(f"{name} must be positive")\n    if config["effective_batch_size"] < 2 or config["effective_batch_size"] % config["batch_size"]:\n        raise ValueError("Use an effective batch >=2 divisible by the physical batch")\n    if config["pool_fraction"] != .2 or config["threshold"] != .5:\n        raise ValueError("Protocol fixes the training pool to 20% and the threshold to 0.5")\n    if config["gumbel_tau"] <= 0 or config["workers"] < 0:\n        raise ValueError("Invalid temperature/worker count")\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef object_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()\n\n\ndef atomic_torch(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    with temporary.open("wb") as handle:\n        torch.save(value, handle)\n        handle.flush()\n        os.fsync(handle.fileno())\n    temporary.replace(path)\n\n\ndef csv_write(path, rows):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not rows:\n        raise ValueError("Refusing an empty manifest/metrics table")\n    columns = list(dict.fromkeys(key for row in rows for key in row))\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=columns)\n        writer.writeheader()\n        writer.writerows(rows)\n    temporary.replace(path)\n\n\ndef pixel_hash(image):\n    return hashlib.sha256(np.asarray(image).tobytes() + str(image.size).encode()).hexdigest()\n\n\ndef _file_map(folder, masks=False):\n    result = {}\n    for path in sorted(Path(folder).rglob("*")):\n        if not path.is_file() or path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:\n            continue\n        match = re.search(r"ISIC_\\d+", path.stem, re.I)\n        if not match:\n            continue\n        identity = match[0].upper()\n        if identity in result:\n            raise ValueError(f"Duplicate image/mask identity {identity} in {folder}")\n        result[identity] = path.resolve()\n    return result\n\n\ndef resolve_dataset_root(reference):\n    if str(reference).strip():\n        root = Path(reference).expanduser().resolve()\n        if not root.is_dir():\n            raise FileNotFoundError(f"ISIC_ROOT does not exist: {root}")\n        return root\n    matches = []\n    for current, dirs, _files in os.walk("/kaggle/input"):\n        dirs[:] = [name for name in dirs if name.lower() not in {"images", "masks", ".cache", ".git"}]\n        if len(Path(current).parts) > 9:\n            dirs[:] = []\n        names = {name.lower() for name in dirs}\n        if {"train", "val", "test"}.issubset(names) or {"train", "valid", "test"}.issubset(names):\n            matches.append(Path(current))\n        elif (any(name.lower().startswith("isic-2017_training_data") for name in dirs)\n              and any(name.lower().startswith("isic-2017_validation_data") for name in dirs)\n              and any(name.lower().startswith("isic-2017_test") and "data" in name.lower() for name in dirs)):\n            matches.append(Path(current))\n    if len(matches) != 1:\n        raise FileNotFoundError(f"Set ISIC_ROOT to one exact ISIC-2017 dataset root; found {matches}")\n    return matches[0].resolve()\n\n\ndef resolve_dino_assets(repo_reference, weights_reference, config):\n    repo, weights = Path(repo_reference).expanduser(), Path(weights_reference).expanduser()\n    filename = f\'{config["backbone"]}_pretrain_lvd1689m-{net.BACKBONES[config["backbone"]][3]}.pth\'\n    if weights.is_dir():\n        matches = list(weights.rglob(filename))\n        if len(matches) != 1:\n            raise FileNotFoundError(f"Need one exact {filename} in {weights}")\n        weights = matches[0]\n    if not repo.is_dir() or not weights.is_file():\n        repos, checkpoints = [], []\n        for current, dirs, files in os.walk("/kaggle/input"):\n            dirs[:] = [name for name in dirs if name.lower() not in {"images", "masks", ".cache", ".git", "__pycache__"}]\n            if len(Path(current).parts) > 10:\n                dirs[:] = []\n            if "backbones.py" in files and Path(current).parts[-2:] == ("dinov3", "hub"):\n                repos.append(Path(current).parents[1])\n            if filename in files:\n                checkpoints.append(Path(current) / filename)\n        if not repo.is_dir():\n            if len(repos) != 1:\n                raise FileNotFoundError(f"Set DINOV3_REPO_DIR explicitly; found {repos}")\n            repo = repos[0]\n        if not weights.is_file():\n            if len(checkpoints) != 1:\n                raise FileNotFoundError(f"Set DINOV3_WEIGHTS to {filename}; found {checkpoints}")\n            weights = checkpoints[0]\n    if not (repo / "dinov3/hub/backbones.py").is_file() or weights.name != filename:\n        raise ValueError("Wrong DINOv3 source/checkpoint; refusing another backbone")\n    return repo.resolve(), weights.resolve()\n\n\ndef load_isic(reference, config, groups_csv=""):\n    root = resolve_dataset_root(reference)\n    frames = {}\n    counts = {"train": 2000, "val": 150, "test": 600}\n    all_ids, cross_split_hashes = set(), {}\n    for role, aliases, official in (("train", ("train", "training"), "Training"),\n                                     ("val", ("val", "valid", "validation"), "Validation"),\n                                     ("test", ("test", "testing"), "Test")):\n        children = {child.name.lower(): child for child in root.iterdir() if child.is_dir()}\n        prepared = [children[name] for name in aliases if name in children]\n        if len(prepared) == 1:\n            base = prepared[0]\n            images, masks = base / "images", base / "masks"\n        elif not prepared:\n            versions = [official, "Test_v2"] if role == "test" else [official]\n            pairs = [(root / f"ISIC-2017_{version}_Data", root / f"ISIC-2017_{version}_Part1_GroundTruth")\n                     for version in versions]\n            found = [pair for pair in pairs if pair[0].is_dir() and pair[1].is_dir()]\n            if len(found) > 1:\n                raise ValueError(f"Ambiguous original ISIC split folders: {found}")\n            images, masks = found[0] if found else pairs[0]\n        else:\n            raise ValueError(f"Ambiguous {role} split folders: {prepared}")\n        if not images.is_dir() or not masks.is_dir():\n            raise FileNotFoundError(f"Need {role} image/mask folders: {images} and {masks}")\n        image_map, mask_map = _file_map(images), _file_map(masks, True)\n        if not image_map or set(image_map) != set(mask_map):\n            raise ValueError(f"Unpaired {role} image/mask IDs: images={len(image_map)}, masks={len(mask_map)}")\n        if config["strict_official_counts"] and len(image_map) != counts[role]:\n            raise ValueError(f"Official ISIC2017 {role} should have {counts[role]} images; found {len(image_map)}")\n        rows = []\n        for image_id, image_path in tqdm(image_map.items(), desc=f"Verify ISIC {role}"):\n            if image_id in all_ids:\n                raise ValueError(f"Image ID overlaps official splits: {image_id}")\n            all_ids.add(image_id)\n            with Image.open(image_path) as source:\n                if source.getexif().get(274, 1) not in {None, 1}:\n                    raise ValueError("Dataset image has EXIF rotation; align its mask explicitly before using it")\n                image = source.convert("RGB")\n            with Image.open(mask_map[image_id]) as mask:\n                if mask.size != image.size:\n                    raise ValueError(f"Image/mask dimensions differ: {image_id}")\n                values = np.unique(np.asarray(mask.convert("L")))\n                if not set(values).issubset({0, 1, 255}):\n                    raise ValueError(f"Expected binary Part1 lesion mask for {image_id}")\n            digest = pixel_hash(image)\n            if digest in cross_split_hashes and cross_split_hashes[digest] != role:\n                raise ValueError(f"Identical image pixels occur in different splits: {image_id}")\n            cross_split_hashes[digest] = role\n            rows.append({"image_id": image_id, "image_path": str(image_path), "mask_path": str(mask_map[image_id]),\n                "split": role, "image_sha256": io.file_sha256(image_path), "mask_sha256": io.file_sha256(mask_map[image_id]),\n                "pixel_sha256": digest, "group_id": ""})\n        frames[role] = rows\n    if groups_csv:\n        with Path(groups_csv).open(encoding="utf-8-sig", newline="") as handle:\n            groups = list(csv.DictReader(handle))\n        group_map = {row["image_id"]: row["group_id"].strip() for row in groups}\n        if len(group_map) != len(groups) or set(group_map) != all_ids or not all(group_map.values()):\n            raise ValueError("Group metadata must supply one nonempty group_id per ISIC image_id")\n        groups_seen = {}\n        for role, rows in frames.items():\n            for row in rows:\n                row["group_id"] = group_map[row["image_id"]]\n                if row["group_id"] in groups_seen and groups_seen[row["group_id"]] != role:\n                    raise ValueError("Patient/group overlaps official splits; resolve the protocol before training")\n                groups_seen[row["group_id"]] = role\n    rng = np.random.default_rng(config["seed"])\n    positions = sorted(rng.choice(len(frames["train"]), max(2, round(.2 * len(frames["train"]))), replace=False).tolist())\n    pool = [dict(frames["train"][index]) for index in positions]\n    return frames, pool\n\n\ndef allowed_candidates(rows, pool, device):\n    allowed = [[q["image_id"] != c["image_id"] and q["pixel_sha256"] != c["pixel_sha256"]\n                and not (q.get("group_id") and q["group_id"] == c.get("group_id")) for c in pool] for q in rows]\n    mask = torch.tensor(allowed, dtype=torch.bool, device=device)\n    if not mask.any(1).all():\n        raise ValueError("Self/group exclusion leaves a query without context")\n    return mask\n\n\nclass SegmentationDataset(Dataset):\n    def __init__(self, rows, config, training=False):\n        self.rows, self.config, self.training = rows, config, training\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        image = io.read_rgb(row["image_path"])\n        with Image.open(row["mask_path"]) as source:\n            mask = source.convert("L")\n        if self.training:\n            scale = random.uniform(.8, 1.) ** .5\n            width, height = max(1, round(image.width * scale)), max(1, round(image.height * scale))\n            left, top = random.randint(0, image.width - width), random.randint(0, image.height - height)\n            box = (left, top, left + width, top + height)\n            image, mask = image.crop(box), mask.crop(box)\n            for method in (Image.Transpose.FLIP_LEFT_RIGHT, Image.Transpose.FLIP_TOP_BOTTOM):\n                if random.random() < .5:\n                    image, mask = image.transpose(method), mask.transpose(method)\n        pixels = io.preprocess(image, self.config)\n        mask = mask.resize((self.config["image_size"],) * 2, Image.Resampling.NEAREST)\n        target = torch.from_numpy((np.asarray(mask).copy() > 0).astype(np.float32))[None]\n        return pixels, target, row\n\n\ndef collate(items):\n    images, masks, rows = zip(*items)\n    return torch.stack(images), torch.stack(masks), list(rows)\n\n\ndef seed_worker(_worker_id):\n    seed = torch.initial_seed() % 2**32\n    random.seed(seed)\n    np.random.seed(seed)\n\n\ndef make_loader(rows, config, training=False, generator=None):\n    return DataLoader(SegmentationDataset(rows, config, training),\n        batch_size=config["batch_size"] if training else config["eval_batch_size"], shuffle=training,\n        num_workers=config["workers"], collate_fn=collate, pin_memory=torch.cuda.is_available(),\n        worker_init_fn=seed_worker, generator=generator, persistent_workers=False)\n\n\n@torch.no_grad()\ndef build_bank(model, pool, config, destination, backbone_sha):\n    identity = object_hash({"backbone": backbone_sha, "image_size": config["image_size"],\n        "pool": [(r["image_id"], r["image_sha256"]) for r in pool], "dtype": config["pool_token_dtype"]})\n    path = Path(destination)\n    if path.is_file():\n        saved = torch.load(path, map_location="cpu", weights_only=True)\n        if saved["identity"] != identity or saved["image_ids"] != [r["image_id"] for r in pool]:\n            raise ValueError("Candidate cache identity changed; preserve the old run and use another output")\n        return saved\n    device = next(model.parameters()).device\n    features, tokens = [], []\n    model.eval()\n    for start in tqdm(range(0, len(pool), config["feature_batch_size"]), desc="Frozen candidate bank"):\n        images = torch.stack([io.preprocess(r["image_path"], config) for r in pool[start:start + config["feature_batch_size"]]]).to(device)\n        initial = model.prepare(images)\n        values, _ = model.encode_selector_tokens(initial)\n        tokens.append(initial.cpu().to(getattr(torch, config["pool_token_dtype"])))\n        features.append(values.cpu())\n    saved = {"identity": identity, "image_ids": [r["image_id"] for r in pool],\n             "tokens": torch.cat(tokens), "features": torch.cat(features)}\n    atomic_torch(path, saved)\n    return saved\n\n\nclass TextTable:\n    def __init__(self, path):\n        self.data = torch.load(path, map_location="cpu", weights_only=True)\n        self.positions = {image_id: index for index, image_id in enumerate(self.data["image_ids"])}\n        if len(self.positions) != len(self.data["image_ids"]):\n            raise ValueError("Duplicate text IDs")\n        values, valid = self.data["embeddings"], self.data["valid"]\n        if not torch.isfinite(values).all() or torch.count_nonzero(values[~valid]):\n            raise ValueError("Invalid captions must be finite zero vectors")\n\n    def gather(self, rows, device):\n        positions = [self.positions[row["image_id"]] for row in rows]\n        return self.data["embeddings"][positions].to(device), self.data["valid"][positions].to(device)\n\n\ndef amp_context(device):\n    if device.type != "cuda":\n        return nullcontext()\n    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    return torch.autocast("cuda", dtype=dtype)\n\n\ndef _selection_inputs(model, rows, pool, bank, table, device):\n    allowed = allowed_candidates(rows, pool, device)\n    q_text = q_valid = c_text = c_valid = None\n    if table is not None:\n        q_text, q_valid = table.gather(rows, device)\n        c_text, c_valid = table.gather(pool, device)\n    return allowed, q_text, q_valid, c_text, c_valid\n\n\n@torch.no_grad()\ndef evaluate(model, loader, pool, bank, config, table=None, desc="validation"):\n    model.eval()\n    device = next(model.parameters()).device\n    totals = dict(loss=0., dice=0., iou=0., q_gate=0., c_gate=0.)\n    predictions = []\n    count = 0\n    for images, masks, rows in tqdm(loader, desc=desc, leave=False):\n        images, masks = images.to(device), masks.to(device)\n        initial = model.prepare(images)\n        global_query, _ = model.encode_selector_tokens(initial)\n        allowed, qt, qv, ct, cv = _selection_inputs(model, rows, pool, bank, table, device)\n        scores = model.score(global_query, bank["features"].to(device), allowed, qt, qv, ct, cv)\n        positions = scores.argmax(1)\n        context = bank["tokens"][positions.cpu()].to(device, dtype=torch.float32)\n        logits, details = model.decode(initial, context, qt, qv,\n            ct[positions] if ct is not None else None, cv[positions] if cv is not None else None)\n        losses = net.segmentation_loss(logits, masks)\n        dice, iou = net.binary_metrics(logits, masks, config["threshold"])\n        for key, values in {"loss": losses, "dice": dice, "iou": iou, **details}.items():\n            if key in totals:\n                totals[key] += float(values.sum())\n        count += len(rows)\n        for index, row in enumerate(rows):\n            predictions.append({"image_id": row["image_id"], "dice": float(dice[index]), "iou": float(iou[index]),\n                "context_id": pool[int(positions[index])]["image_id"], "context_position": int(positions[index]),\n                "q_gate": float(details["q_gate"][index]), "c_gate": float(details["c_gate"][index])})\n    return {key: value / count for key, value in totals.items()}, predictions\n\n\ndef train_epoch(model, loader, pool, bank, table, optimizer, scaler, config, epoch):\n    model.train()\n    device = next(model.parameters()).device\n    totals = {key: 0. for key in ("loss", "task_loss", "policy_loss", "reward", "dice", "iou", "q_gate", "c_gate")}\n    count = 0\n    iterator = iter(loader)\n    steps = math.ceil(config["effective_batch_size"] / config["batch_size"])\n    progress = tqdm(total=len(loader.dataset), desc=f"Epoch {epoch:03d} train", unit="image")\n    while True:\n        group = []\n        for _ in range(steps):\n            batch = next(iterator, None)\n            if batch is None:\n                break\n            images, masks, rows = batch\n            images, masks = images.to(device), masks.to(device)\n            with torch.no_grad():\n                initial = model.prepare(images)\n                global_query, _ = model.encode_selector_tokens(initial)\n                allowed, qt, qv, ct, cv = _selection_inputs(model, rows, pool, bank, table, device)\n                scores = model.score(global_query, bank["features"].to(device), allowed, qt, qv, ct, cv)\n                actions = torch.distributions.Categorical(logits=scores).sample()\n                context = bank["tokens"][actions.cpu()].to(device, dtype=torch.float32)\n                with amp_context(device):\n                    # Query text is identical in BOTH reward terms. Only context\n                    # image + its own caption is removed from the baseline.\n                    alone, _ = model.decode(initial, None, qt, qv)\n                    paired, _ = model.decode(initial, context, qt, qv,\n                        ct[actions] if ct is not None else None, cv[actions] if cv is not None else None)\n                reward = net.segmentation_loss(alone, masks) - net.segmentation_loss(paired, masks)\n            group.append((initial.detach(), global_query.detach(), masks, rows, allowed, qt, qv, ct, cv, actions, reward))\n        if not group:\n            break\n        # Standardize over the EFFECTIVE batch, not a physical batch of size 1.\n        # No model/optimizer update occurs between policy sampling and log_prob.\n        rewards = torch.cat([item[-1] for item in group])\n        advantages = ((rewards - rewards.mean()) / rewards.std(unbiased=False).clamp_min(1e-6)).detach()\n        optimizer.zero_grad(set_to_none=True)\n        offset = 0\n        for initial, global_query, masks, rows, allowed, qt, qv, ct, cv, actions, reward in group:\n            scores = model.score(global_query, bank["features"].to(device), allowed, qt, qv, ct, cv)\n            policy = torch.distributions.Categorical(logits=scores)\n            weights = torch.nn.functional.gumbel_softmax(scores, tau=config["gumbel_tau"], hard=True, dim=1)\n            context = net.select_bank(weights, bank["tokens"])\n            c_text = weights @ ct if ct is not None else None\n            c_valid = weights @ cv.float() if cv is not None else None\n            with amp_context(device):\n                logits, details = model.decode(initial, context, qt, qv, c_text, c_valid)\n                task_loss = net.segmentation_loss(logits, masks).mean()\n            policy_loss = -(policy.log_prob(actions) * advantages[offset:offset + len(rows)]).mean()\n            loss = task_loss + config["policy_weight"] * policy_loss - config["entropy_weight"] * policy.entropy().mean()\n            if not torch.isfinite(loss):\n                raise FloatingPointError("Non-finite training objective; no checkpoint exported")\n            scaler.scale(loss * len(rows) / len(rewards)).backward()\n            dice, iou = net.binary_metrics(logits.detach(), masks, config["threshold"])\n            for key, value in {"loss": loss, "task_loss": task_loss, "policy_loss": policy_loss,\n                "reward": reward.mean(), "dice": dice.mean(), "iou": iou.mean(),\n                "q_gate": details["q_gate"].mean(), "c_gate": details["c_gate"].mean()}.items():\n                totals[key] += float(value.detach()) * len(rows)\n            offset += len(rows)\n            count += len(rows)\n            progress.update(len(rows))\n        scaler.unscale_(optimizer)\n        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], config["grad_clip"], error_if_nonfinite=True)\n        scaler.step(optimizer)\n        scaler.update()\n        del group\n    progress.close()\n    return {key: value / count for key, value in totals.items()}\n\n\ndef run_training(model, frames, pool, bank, config, output, identity, table=None):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    protocol = output / "protocol.json"\n    if protocol.is_file() and io.read_json(protocol)["identity"] != identity:\n        raise ValueError("Run identity/config/data/source changed; do not overwrite or reuse old checkpoints")\n    io.write_json(protocol, {"identity": identity, "config": config})\n    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=config["lr"], weight_decay=config["weight_decay"])\n    def schedule(epoch):\n        if epoch < config["warmup_epochs"]:\n            return (epoch + 1) / max(1, config["warmup_epochs"])\n        t = (epoch - config["warmup_epochs"]) / max(1, config["epochs"] - config["warmup_epochs"])\n        return .5 * (1 + math.cos(math.pi * min(t, 1)))\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)\n    device = next(model.parameters()).device\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not torch.cuda.is_bf16_supported())\n    generator = torch.Generator().manual_seed(config["seed"])\n    history, start, best, bad = [], 1, -1., 0\n    if (output / "last.pt").is_file():\n        saved = torch.load(output / "last.pt", map_location="cpu", weights_only=True)\n        if saved["identity"] != identity:\n            raise ValueError("Wrong resume checkpoint identity")\n        model.restore_trainable(saved["state"])\n        optimizer.load_state_dict(saved["optimizer"])\n        scheduler.load_state_dict(saved["scheduler"])\n        scaler.load_state_dict(saved["scaler"])\n        history, start, best, bad = saved["history"], saved["epoch"] + 1, saved["best_dice"], saved["bad_epochs"]\n        random.setstate(saved["python_rng"])\n        np_state = saved["numpy_rng"]\n        np.random.set_state((np_state[0], np.array(np_state[1], dtype=np.uint32), *np_state[2:]))\n        torch.set_rng_state(saved["torch_rng"])\n        if device.type == "cuda":\n            torch.cuda.set_rng_state_all(saved["cuda_rng"])\n        generator.set_state(saved["loader_rng"])\n    train_loader = make_loader(frames["train"], config, True, generator)\n    val_loader = make_loader(frames["val"], config)\n    for epoch in range(start, config["epochs"] + 1):\n        if config["patience"] and bad >= config["patience"]:\n            break\n        train = train_epoch(model, train_loader, pool, bank, table, optimizer, scaler, config, epoch)\n        val, _ = evaluate(model, val_loader, pool, bank, config, table)\n        history.append({"epoch": epoch, "lr": optimizer.param_groups[0]["lr"],\n                        **{f"train_{k}": v for k, v in train.items()}, **{f"val_{k}": v for k, v in val.items()}})\n        if val["dice"] > best:\n            best, bad = val["dice"], 0\n            atomic_torch(output / "best.pt", {"identity": identity, "epoch": epoch, "config": config,\n                "validation": val, "state": model.trainable_state()})\n        else:\n            bad += 1\n        scheduler.step()\n        numpy_rng = np.random.get_state()\n        atomic_torch(output / "last.pt", {"identity": identity, "epoch": epoch, "best_dice": best, "bad_epochs": bad,\n            "state": model.trainable_state(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),\n            "scaler": scaler.state_dict(), "history": history, "python_rng": random.getstate(),\n            "numpy_rng": (numpy_rng[0], numpy_rng[1].tolist(), *numpy_rng[2:]), "torch_rng": torch.get_rng_state(),\n            "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else [], "loader_rng": generator.get_state()})\n        csv_write(output / "history.csv", history)\n        print(f\'{config["mode"]} epoch={epoch} val_dice={val["dice"]:.4f} val_iou={val["iou"]:.4f} \'\n              f\'q_gate={val["q_gate"]:.4f} c_gate={val["c_gate"]:.4f}\', flush=True)\n    best_state = torch.load(output / "best.pt", map_location="cpu", weights_only=True)\n    if best_state["identity"] != identity:\n        raise ValueError("Best checkpoint belongs to another run")\n    model.restore_trainable(best_state["state"])\n    model.eval()\n    return history, best_state\n\n\ndef source_identity(folder):\n    folder = Path(folder)\n    files = {p.relative_to(folder).as_posix(): io.file_sha256(p) for p in sorted(folder.rglob("*.py"))\n             if not {".git", "__pycache__", ".cache"}.intersection(p.relative_to(folder).parts)}\n    return object_hash(files)\n\n\ndef export_bundle(model, frames, pool, bank, config, output, repo, weights, runtime_dir, metrics,\n                  *, caption_assets=None, caption_output=None, cold_verify=True, factory=None):\n    """Ready is written only after strict state/key/hash, reload and attention parity."""\n    output = Path(output)\n    # Only completion metadata is needed here, not a second CPU copy of all\n    # Adam moments. Restore tensor storage on meta to keep export RAM bounded.\n    last = torch.load(output / "last.pt", map_location="meta", weights_only=True)\n    if last["epoch"] < config["epochs"] and not (config["patience"] and last["bad_epochs"] >= config["patience"]):\n        raise RuntimeError("Training is incomplete; resume before final deployment export")\n    best_path = output / "best.pt"\n    best = torch.load(best_path, map_location="cpu", weights_only=True)\n    model.restore_trainable(best["state"])\n    model.eval()\n    root = output / "exports" / f\'isic2017_{config["mode"]}_vitl16_448_web_bundle\'\n    root.mkdir(parents=True, exist_ok=True)\n    io.write_json(root / "export_status.json", {"status": "staging"})\n    if (root / "manifest.json").exists():\n        (root / "manifest.json").unlink()\n    (root / "models").mkdir(exist_ok=True)\n    (root / "runtime").mkdir(exist_ok=True)\n    io.write_json(root / "config.json", config)\n    state = model.trainable_state()\n    is_selector = lambda key: key.startswith("selector.") or key == "semantic_mix_logit"\n    atomic_torch(root / "models/selector.pt", {key: value for key, value in state.items() if is_selector(key)})\n    atomic_torch(root / "models/segmentor.pt", {key: value for key, value in state.items() if not is_selector(key)})\n    backbone_file = "models/" + Path(weights).name\n    shutil.copy2(weights, root / backbone_file)\n    atomic_torch(root / "candidate_bank.pt", bank)\n    shutil.copytree(repo, root / "vendor/dinov3", dirs_exist_ok=True,\n        ignore=shutil.ignore_patterns(".git", "__pycache__", ".cache", "*.pyc", "*.pth", "*.pt", "*.ipynb", "data", "outputs"))\n    for name in ("isic_segmentation_web.py", "tacs_visual_web.py", "train_isic_segmentation_web.py",\n                 "isic_caption_web.py", "isic_caption_core.py"):\n        path = Path(runtime_dir) / name\n        if path.is_file():\n            shutil.copy2(path, root / "runtime" / name)\n    exported_pool = []\n    for index, row in enumerate(tqdm(pool, desc="Export real context images")):\n        target = root / "candidate_images" / (row["image_id"] + Path(row["image_path"]).suffix.lower())\n        target.parent.mkdir(exist_ok=True)\n        # Preserve real source bytes; converting high-resolution dermoscopy\n        # JPEGs to RGB PNG unnecessarily expands the deployment by gigabytes.\n        shutil.copy2(row["image_path"], target)\n        exported_pool.append({"image_id": row["image_id"], "position": index, "image": target.relative_to(root).as_posix(),\n            "pixel_sha256": row["pixel_sha256"], "group_id": row.get("group_id", ""), "split": "train"})\n    io.write_json(root / "candidate_pool.json", exported_pool)\n    gt_index, caption_index = {}, {}\n    for role, rows in frames.items():\n        portable = []\n        for row in rows:\n            target = root / "ground_truth_masks" / f\'{row["image_id"]}.png\'\n            target.parent.mkdir(exist_ok=True)\n            with Image.open(row["mask_path"]) as image:\n                Image.fromarray((np.asarray(image.convert("L")) > 0).astype(np.uint8) * 255).save(target)\n            gt_index[row["pixel_sha256"]] = {"image_id": row["image_id"], "split": role,\n                "mask": target.relative_to(root).as_posix(), "group_id": row.get("group_id", "")}\n            caption_index[row["pixel_sha256"]] = row["image_id"]\n            portable.append({key: value for key, value in row.items() if key not in {"image_path", "mask_path"}})\n        csv_write(root / f"{role}_split.csv", portable)\n    io.write_json(root / "ground_truth_index.json", gt_index)\n    example = frames["val"][0]\n    (root / "example").mkdir(exist_ok=True)\n    io.read_rgb(example["image_path"]).save(root / "example/query.png")\n    shutil.copy2(output / "history.csv", root / "history.csv")\n    io.write_json(root / "metrics.json", metrics)\n    io.write_json(root / "training_identity.json", {"identity": best["identity"], "epoch": best["epoch"],\n        "best_checkpoint_sha256": io.file_sha256(best_path), "validation": best["validation"],\n        "patient_groups_available": bool(frames["train"][0].get("group_id")), "source": source_identity(runtime_dir)})\n    if config["mode"] == "mtacs":\n        if not caption_assets or not caption_output:\n            raise ValueError("M-TACS export requires actual Florence/CLIP assets and registered captions")\n        for key in ("florence", "clip"):\n            shutil.copytree(caption_assets[key], root / "caption_models" / key, dirs_exist_ok=True,\n                ignore=shutil.ignore_patterns(".cache", ".git", "__pycache__", "*.pyc"))\n        caption_output = Path(caption_output)\n        shutil.copy2(caption_output / "captions.csv", root / "captions.csv")\n        shutil.copy2(caption_output / "text_cache/clip_text.pt", root / "text_embeddings.pt")\n        with (root / "captions.csv").open(encoding="utf-8", newline="") as handle:\n            lookup = {row["image_id"]: row for row in csv.DictReader(handle)}\n        io.write_json(root / "caption_lookup.json", lookup)\n        shutil.copy2(caption_output / "web_caption_settings.json", root / "caption_settings.json")\n        for directory in ("generation_protocol", "text_cache"):\n            for name in ("protocol.json", "status.json"):\n                path = caption_output / directory / name\n                if path.is_file():\n                    target = root / "caption_provenance" / directory / name\n                    target.parent.mkdir(parents=True, exist_ok=True)\n                    shutil.copy2(path, target)\n        wheels = caption_output / "caption_runtime/wheels"\n        if wheels.is_dir():\n            shutil.copytree(wheels, root / "offline_wheels", dirs_exist_ok=True)\n        from isic_caption_web import probe_export\n        fresh_caption_check = probe_export(root, "cpu", caption_assets["offline_assets_root"], output / "export_caption_runtime")\n        io.write_json(root / "example/fresh_caption_check.json", fresh_caption_check)\n    versions = {"python": sys.version, "torch": str(torch.__version__), "numpy": np.__version__,\n                "device": str(next(model.parameters()).device)}\n    io.write_json(root / "environment.json", versions)\n    (root / "requirements.txt").write_text("torch>=2.5\\nnumpy\\nPillow\\ntqdm\\n# M-TACS caption child only:\\ntransformers==4.49.0\\naccelerate\\neinops\\ntimm\\n", encoding="utf-8")\n    readme = f\'\'\'# ISIC2017 {config["mode"]}: complete segmentation deployment bundle\n\nBackbone: {config["backbone"]}, frozen; image size {config["image_size"]}; one context from the fixed training-only pool.\nModel parts: models/ official DINO checkpoint + selector.pt + segmentor.pt. All three are required.\nThe segmentor component includes EVERY per-block context module, DPT, and M-TACS query/candidate text projectors/gates.\ncandidate_bank.pt is aligned exactly to candidate_pool.json and the actual candidate_images/.\nGround-truth masks are DISPLAY/EVALUATION annotations, never context/model inputs.\n\nPython: add runtime/ to sys.path, import isic_segmentation_web, construct Predictor(bundle_path, device).\nUse .predict(image) for the mask, probability array, overlay, selected candidate and attention.\nFor new M-TACS images, pass caption_engine=BundledCaptionEngine(bundle_path, device) from isic_caption_web\nin a separate compatible Transformers 4.49 worker. Cached known images can use verified pixel bindings.\nCLI: python runtime/isic_segmentation_web.py --bundle . --image example/query.png --output prediction --device cpu\nAdd --captions for a new M-TACS image when running inside the compatible caption environment.\nThe environment must supply compatible PyTorch/Pillow/NumPy; this ZIP does not contain a Python interpreter.\nOriginal model/source/dataset licenses remain applicable. Vendor source includes its original license files.\n\nAttention is last-block CLS-to-patch, averaged over heads. It is not Grad-CAM or a causal/diagnostic claim.\nThe selector map describes its frozen image encoder; the learned utility projections and semantic score are not token explanations.\nThe binary mask is the model\'s lesion-region prediction, not a diagnosis or a calibrated probability of malignancy.\nThis is a declared reconstruction of the published architecture, with an explicitly implemented lightweight DPT\nand learned projection utility head. It is not a claim of byte-identical unpublished author implementation.\nPaper settings were S/16, 256, 3 seeds. This run\'s exact settings and metrics are in config.json / metrics.json.\nTraining best.pt/last.pt remain OUTSIDE this ZIP; preserve the full run directory to resume training.\n\'\'\'\n    (root / "README.md").write_text(readme, encoding="utf-8")\n    manifest = {"schema": net.SCHEMA, "status": "staged", "mode": config["mode"], "backbone_file": backbone_file,\n                "caption_pixel_index": caption_index if config["mode"] == "mtacs" else {}, "files": {}}\n    def bind_files():\n        manifest["files"] = {p.relative_to(root).as_posix(): io.file_sha256(p) for p in sorted(root.rglob("*"))\n            if p.is_file() and p.name not in {"manifest.json", "export_status.json"} and "__pycache__" not in p.parts}\n        io.write_json(root / "manifest.json", manifest)\n    bind_files()\n    # Independent model construction + split checkpoint reload, then attention\n    # invariance and a cold-process replay without original Input paths.\n    predictor = net.Predictor(root, "cpu", factory=factory, _allow_staged=True)\n    plain = predictor.predict(root / "example/query.png", return_attention=False)\n    explained = predictor.predict(root / "example/query.png", return_attention=True)\n    if not torch.allclose(plain["logits"], explained["logits"], atol=1e-5, rtol=1e-5):\n        raise ValueError("Attention capture changed segmentation logits")\n    # Compare against the trained in-memory model on the SAME quantized bank.\n    device = next(model.parameters()).device\n    images = io.preprocess(root / "example/query.png", config)[None].to(device)\n    qinitial = model.prepare(images)\n    qglobal, _ = model.encode_selector_tokens(qinitial)\n    table = TextTable(root / "text_embeddings.pt") if config["mode"] == "mtacs" else None\n    allowed, qt, qv, ct, cv = _selection_inputs(model, [example], pool, bank, table, device)\n    with torch.no_grad():\n        scores = model.score(qglobal, bank["features"].to(device), allowed, qt, qv, ct, cv)\n        chosen = scores.argmax(1)\n        original, _ = model.decode(qinitial, bank["tokens"][chosen.cpu()].to(device, dtype=torch.float32), qt, qv,\n            ct[chosen] if ct is not None else None, cv[chosen] if cv is not None else None)\n    if int(chosen) != plain["evidence"]["candidate"]["position"] or not torch.allclose(original.cpu(), plain["logits"], atol=5e-4, rtol=5e-4):\n        raise ValueError("Reloaded export differs from the trained best model")\n    atomic_torch(root / "example/expected_logits.pt", plain["logits"])\n    np.save(root / "example/probabilities.npy", explained.pop("probabilities"))\n    explained.pop("logits")\n    io.write_json(root / "example/prediction.json", explained)\n    del predictor\n    bind_files()\n    if cold_verify:\n        env = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")\n        env.pop("PYTHONPATH", None)\n        command = [sys.executable, str(root / "runtime/isic_segmentation_web.py"), "--bundle", str(root),\n            "--image", str(root / "example/query.png"), "--output", str(output / "cold_reload_check"), "--device", "cpu", "--verify-staged"]\n        completed = subprocess.run(command, cwd=root, env=env, text=True, capture_output=True)\n        if completed.returncode:\n            raise RuntimeError("Cold offline reload failed: " + completed.stderr[-5000:])\n        cold = torch.load(output / "cold_reload_check/logits.pt", map_location="cpu", weights_only=True)\n        if not torch.allclose(cold, plain["logits"], atol=1e-5, rtol=1e-5):\n            raise ValueError("Cold-process mask logits differ")\n    manifest["status"] = "ready" if cold_verify else "fixture_unverified"\n    manifest["verification"] = {"state_reload": True, "attention_parity": True, "trained_model_parity": True,\n                                "cold_offline_reload": cold_verify}\n    bind_files()\n    io.write_json(root / "export_status.json", {"status": manifest["status"]})\n    if not cold_verify:\n        return root, None\n    archive = root.with_suffix(".zip")\n    with zipfile.ZipFile(archive.with_suffix(".zip.tmp"), "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zipped:\n        for path in sorted(root.rglob("*")):\n            if path.is_file() and "__pycache__" not in path.parts:\n                zipped.write(path, path.relative_to(root.parent).as_posix())\n    archive.with_suffix(".zip.tmp").replace(archive)\n    io.write_json(output / "export_receipt.json", {"bundle": str(root), "zip": str(archive),\n        "zip_sha256": io.file_sha256(archive), "best_sha256": io.file_sha256(best_path), "status": "ready"})\n    return root, archive\n', 'isic_caption_core.py': 'from __future__ import annotations\nimport csv\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nimport re\nimport shutil\nimport struct\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\n\ndef atomic_csv(path, rows, columns):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n        handle.flush()\n        os.fsync(handle.fileno())\n    temporary.replace(path)\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\ndef checkpoint_asset_identity(model_dir):\n    model_dir = Path(model_dir)\n    files = sorted(path for path in model_dir.iterdir() if path.is_file() and\n                   (path.suffix in {".py", ".json", ".safetensors", ".bin", ".txt"}))\n    if not any(path.suffix in {".safetensors", ".bin"} for path in files):\n        raise ValueError("Checkpoint has no weight files")\n    return {path.name: file_sha256(path) for path in files}\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\ndef object_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True,\n                                    allow_nan=False).encode()).hexdigest()\n\ndef read_csv_rows(path, required=()):\n    with Path(path).open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if not reader.fieldnames or len(set(reader.fieldnames)) != len(reader.fieldnames):\n            raise ValueError(f"Missing/duplicate CSV header: {path}")\n        if not set(required).issubset(reader.fieldnames):\n            raise ValueError(f"Missing required CSV columns: {sorted(set(required) - set(reader.fieldnames))}")\n        rows = list(reader)\n    if any(None in row or any(value is None for value in row.values()) for row in rows):\n        raise ValueError(f"Malformed/incomplete CSV row: {path}; original file was not modified")\n    return rows\n\ndef clean_dtd_caption(text):\n    import re\n    text = str(text).strip()\n    for _ in range(3):\n        before = text\n        text = re.sub(r"^(?:(?:the|this)\\s+)?(?:image|picture|photo(?:graph)?)\\s+"\n                      r"(?:is|shows|depicts|features|contains)\\s+", "", text, flags=re.I)\n        text = re.sub(r"^(?:(?:a|an|the)\\s+)?close[ -]up(?:\\s+view)?\\s+(?:of|showing)\\s+",\n                      "", text, flags=re.I)\n        text = re.sub(r"^(?:a|an|the)(?:\\s+|$)", "", text, flags=re.I)\n        if text == before:\n            break\n    words = re.findall(r"[A-Za-z0-9]+(?:[\'\\u2019][A-Za-z0-9]+)?", text)\n    selected = []\n    normalized_count = 0\n    for word in words:\n        count = len(re.findall(r"[A-Za-z0-9]+", word))\n        if normalized_count + count > 8:\n            break\n        selected.append(word)\n        normalized_count += count\n    return " ".join(selected)\n\ndef available_rows(directory, records, identity):\n    return [row for record in records if (row := cached_row(directory, record, identity)) is not None]\n\ndef cache_path(directory, image_id):\n    return Path(directory) / (object_hash(image_id) + ".json")\n\ndef cached_row(directory, record, identity):\n    path = cache_path(directory, record["image_id"])\n    if not path.is_file():\n        return None\n    result = json.loads(path.read_text(encoding="utf-8"))\n    if (result.get("identity") != identity or result.get("row", {}).get("image_id") != record["image_id"]\n            or result["row"].get("split") != record["split"]):\n        raise ValueError(f"Wrong cached image/protocol: {path}")\n    return result["row"]\n\ndef local_model_configs(roots):\n    seen = set()\n    for root in roots:\n        root = Path(root).expanduser()\n        if not root.is_dir():\n            continue\n        for current, dirs, files in os.walk(root):\n            dirs[:] = sorted(name for name in dirs if name not in {\n                "images", ".git", "__pycache__", "node_modules", ".cache"})\n            if len(Path(current).relative_to(root).parts) >= 9:\n                dirs[:] = []\n            if "config.json" not in files:\n                continue\n            path = Path(current).resolve()\n            if str(path) not in seen:\n                seen.add(str(path))\n                yield path\n\ndef lock_protocol(path, protocol):\n    path = Path(path)\n    if path.is_file():\n        if json.loads(path.read_text(encoding="utf-8")) != protocol:\n            raise ValueError(f"Changed images/model/prompt/runtime/settings. Use a new output directory: {path.parent}")\n    else:\n        if path.parent.exists() and any(path.parent.iterdir()):\n            raise ValueError(f"Refusing to adopt cached output without its protocol: {path.parent}")\n        atomic_json(path, protocol)\n    return object_hash(protocol)\n\ndef qwen_asset_identity(path):\n    result = checkpoint_asset_identity(path)\n    for template in sorted(Path(path).glob("*.jinja")):\n        result[template.name] = file_sha256(template)\n    return result\n\ndef resolve_clip_model(reference, roots):\n    def is_clip(path):\n        try:\n            return json.loads((path / "config.json").read_text(encoding="utf-8")).get("model_type") == "clip"\n        except (OSError, ValueError):\n            return False\n    if str(reference).strip() and Path(reference).is_dir():\n        path = Path(reference).resolve()\n        if not is_clip(path):\n            raise ValueError(f"Not a full local CLIP checkpoint: {path}")\n        return path\n    matches = [path for path in local_model_configs(roots) if is_clip(path)]\n    if len(matches) != 1:\n        raise FileNotFoundError("Set CLIP_MODEL_LOCAL_PATH to the exact full CLIP ViT-B/32 folder")\n    return matches[0]\n\ndef run_worker(worker_path, payload_path, phase, env, log_path):\n    Path(log_path).parent.mkdir(parents=True, exist_ok=True)\n    with Path(log_path).open("a", encoding="utf-8") as log:\n        process = subprocess.Popen([sys.executable, str(worker_path), str(payload_path), phase], env=env,\n            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)\n        try:\n            for line in process.stdout:\n                print(line, end="", flush=True)\n                log.write(line)\n                log.flush()\n            code = process.wait()\n        except BaseException:\n            process.terminate()\n            process.wait()\n            raise\n        finally:\n            process.stdout.close()\n    if code:\n        raise RuntimeError(f"{phase} worker exited with code {code}. See {log_path}; committed image caches are preserved")\n\ndef runtime_versions():\n    result = {}\n    for name in ("torch", "torchvision", "transformers", "accelerate", "tokenizers", "Pillow", "numpy"):\n        result[name] = importlib.metadata.version(name)\n    return result\n\ndef safetensors_parameter_count(files):\n    # Read headers only, not tensor contents; verify the attached model\'s actual scale.\n    count = 0\n    for path in files:\n        with Path(path).open("rb") as handle:\n            raw_length = handle.read(8)\n            if len(raw_length) != 8:\n                raise ValueError(f"Truncated safetensors header: {path}")\n            length = struct.unpack("<Q", raw_length)[0]\n            if not 2 <= length <= 64 * 1024 * 1024 or length + 8 > Path(path).stat().st_size:\n                raise ValueError(f"Invalid safetensors header length: {path}")\n            header = json.loads(handle.read(length))\n        for name, tensor in header.items():\n            if name == "__metadata__":\n                continue\n            if tensor.get("dtype") not in {"F16", "BF16", "F32"}:\n                raise ValueError("Use original floating-point weights, not quantized/FP8 weights, for this comparison")\n            shape = tensor.get("shape", [])\n            if not all(type(size) is int and size >= 0 for size in shape):\n                raise ValueError(f"Invalid tensor shape: {path}: {name}")\n            count += math.prod(shape)\n    return count\n\ndef weight_files(path):\n    path = Path(path).resolve()\n    index = path / "model.safetensors.index.json"\n    if index.is_file():\n        names = set(json.loads(index.read_text(encoding="utf-8")).get("weight_map", {}).values())\n        if not names:\n            raise ValueError(f"Empty safetensors weight index: {index}")\n        if any(not isinstance(name, str) or "/" in name or "\\\\" in name or Path(name).is_absolute() for name in names):\n            raise ValueError(f"Weight index must reference shard filenames inside its model folder: {index}")\n        files = [path / name for name in sorted(names)]\n    else:\n        files = [path / "model.safetensors"]\n    for file in files:\n        # HF snapshot files may be read-only symlinks into a sibling blobs directory.\n        if not file.is_file() or file.suffix != ".safetensors":\n            raise FileNotFoundError(f"Missing/unsafe local safetensors shard: {file}")\n    return files\n\nPOLICY = "isic_visible_attributes_v1"\n\nCLINICAL_PATTERN = r"\\b(?:melanoma|melanomas|melanocytic|nevus|nevi|naevus|naevi|benign|malignant|malignancy|cancer|cancerous|carcinoma|keratosis|keratoses|diagnosis|diagnosed)\\b"\n\nCOLUMNS = ["image_id", "split", "raw_caption", "final_caption", "caption_valid", "leakage_detected",\n           "clinical_term_detected", "word_count", "source_long_caption", "generation_token_limit_reached",\n           "vlm_model", "prompt_version", "caption_input_policy", "captioner_fingerprint", "retry_count"]\n\ndef caption_policy(response, token_limit=False):\n    caption = clean_dtd_caption(response)\n    count = len(re.findall(r"[A-Za-z0-9]+", caption))\n    clinical = bool(re.search(CLINICAL_PATTERN, str(response), re.I))\n    valid = 5 <= count <= 8 and not clinical and not token_limit\n    return caption, valid, clinical, count\n\ndef validate_caption_records(records):\n    if not records or len({row["image_id"] for row in records}) != len(records):\n        raise ValueError("Caption image IDs must be nonempty and unique across splits")\n    for row in records:\n        if set(row) != {"image_id", "image_path", "image_sha256", "split"}:\n            raise ValueError("Caption worker accepts pixels/identity only, never masks, class labels or clinical metadata")\n        if row["split"] not in {"train", "val", "test"}:\n            raise ValueError("Unexpected split")\n        if file_sha256(row["image_path"]) != row["image_sha256"]:\n            raise ValueError(f"Image changed: {row[\'image_id\']}")\n\ndef validate_text_cache(cache, rows, csv_sha256, identity):\n    import torch\n    expected_valid = torch.tensor([row["caption_valid"].lower() == "true" for row in rows], dtype=torch.bool)\n    vectors, valid = cache["embeddings"], cache["valid"]\n    if (cache["identity"] != identity or cache["image_ids"] != [row["image_id"] for row in rows]\n            or cache["caption_csv_sha256"] != csv_sha256 or vectors.ndim != 2 or len(vectors) != len(rows)\n            or valid.dtype != torch.bool or not torch.equal(valid.cpu(), expected_valid)):\n        raise ValueError("Wrong CLIP text cache identity/IDs/shape/masks")\n    if not torch.isfinite(vectors).all() or torch.count_nonzero(vectors[~valid]):\n        raise ValueError("Invalid captions must have finite zero embeddings")\n    if valid.any() and not torch.allclose(vectors[valid].float().norm(dim=1), torch.ones(int(valid.sum())), atol=1e-4):\n        raise ValueError("Valid CLIP caption vectors must have unit norm")\n\nMETHOD = "florence_prefix_control"\n\nMODEL_ID = "microsoft/Florence-2-base"\n\nTASK_PROMPT = "<MORE_DETAILED_CAPTION>"\n\ndef validate_florence_prefix_folder(path):\n    path = Path(path).expanduser().resolve()\n    config = json.loads((path / "config.json").read_text(encoding="utf-8"))\n    if (config.get("model_type") != "florence2"\n            or "AutoModelForCausalLM" not in config.get("auto_map", {})):\n        raise ValueError("Florence-Prefix requires the original legacy Microsoft Florence-2-base checkpoint")\n    required = ("configuration_florence2.py", "modeling_florence2.py", "processing_florence2.py",\n                "preprocessor_config.json", "tokenizer_config.json")\n    if any(not (path / name).is_file() for name in required):\n        raise FileNotFoundError("Incomplete local Florence model code/processor/tokenizer files")\n    if not ((path / "tokenizer.json").is_file() or all((path / name).is_file() for name in ("vocab.json", "merges.txt"))):\n        raise FileNotFoundError("Missing Florence tokenizer vocabulary")\n    if (path / "adapter_config.json").exists() or "-ft" in path.name.lower():\n        raise ValueError("Use the original frozen base model, not a LoRA or -ft captioner")\n    count = safetensors_parameter_count(weight_files(path))\n    if not 180_000_000 <= count <= 300_000_000:\n        raise ValueError(f"Expected Florence-2-base scale, not large/quantized/adapter weights: {count:,}")\n    return path, count\n\nclass IsicFlorencePrefixCaptioner:\n    def __init__(self, path, settings):\n        import torch\n        import transformers\n        from transformers import AutoModelForCausalLM, AutoProcessor\n\n        if not transformers.__version__.startswith("4.49."):\n            raise RuntimeError("Run the legacy Florence worker with the isolated offline Transformers 4.49 environment")\n        self.device = torch.device(settings.get("device", "cuda:0"))\n        self.settings = settings\n        self.dtype = torch.float32\n        self.processor = AutoProcessor.from_pretrained(str(path), trust_remote_code=True, local_files_only=True)\n        model, info = AutoModelForCausalLM.from_pretrained(\n            str(path), torch_dtype=self.dtype, trust_remote_code=True, local_files_only=True,\n            attn_implementation="eager", low_cpu_mem_usage=True, output_loading_info=True,\n        )\n        missing = [key for key in info.get("missing_keys", []) if not key.endswith("position_ids")]\n        if missing or info.get("mismatched_keys"):\n            raise ValueError(f"Incomplete Florence weights: {missing}")\n        self.model = model.to(self.device).eval().requires_grad_(False)\n\n    def generate(self, images):\n        import torch\n\n        inputs = self.processor(text=[TASK_PROMPT] * len(images), images=images,\n                                return_tensors="pt", padding=True)\n        moved = {key: value.to(device=self.device, dtype=self.dtype if key == "pixel_values" else value.dtype)\n                 for key, value in inputs.items() if torch.is_tensor(value)}\n        with torch.inference_mode():\n            output = self.model.generate(**moved, max_new_tokens=self.settings["max_new_tokens"],\n                                         num_beams=1, do_sample=False, use_cache=True, early_stopping=False)\n        texts = self.processor.batch_decode(output, skip_special_tokens=False)\n        if len(texts) != len(images):\n            raise ValueError("Florence response count mismatch")\n        eos = self.model.generation_config.eos_token_id\n        eos = set(eos if isinstance(eos, (list, tuple)) else [eos])\n        responses = []\n        for text, image, sequence in zip(texts, images, output.cpu().tolist()):\n            parsed = self.processor.post_process_generation(text, task=TASK_PROMPT,\n                                                           image_size=(image.width, image.height))\n            if not isinstance(parsed, dict) or not isinstance(parsed.get(TASK_PROMPT), str):\n                raise ValueError("Florence did not return the requested detailed-caption field")\n            # Florence is encoder-decoder: its first decoder token can itself be EOS.\n            new_tokens = sequence[1:]\n            limited = (len(new_tokens) >= self.settings["max_new_tokens"]\n                       and not any(token in eos for token in new_tokens))\n            responses.append((parsed[TASK_PROMPT].strip(), limited))\n        return responses\n\nMODEL_IDS = {METHOD: MODEL_ID}\nCAPTION_PROMPT = TASK_PROMPT\n\ndef make_generation_row(record, response, token_limit, identity):\n    caption, valid, clinical, count = caption_policy(response, token_limit)\n    return {"image_id": record["image_id"], "split": record["split"], "raw_caption": caption,\n            "final_caption": caption, "caption_valid": valid, "leakage_detected": clinical,\n            "clinical_term_detected": clinical, "word_count": count, "source_long_caption": response,\n            "generation_token_limit_reached": token_limit, "vlm_model": MODEL_IDS[METHOD],\n            "prompt_version": POLICY, "caption_input_policy": POLICY,\n            "captioner_fingerprint": identity, "retry_count": 0}\n\ndef generate_isic_records(payload, identity, factory=IsicFlorencePrefixCaptioner):\n    import torch\n    from PIL import Image, ImageOps\n    from tqdm.auto import tqdm\n    records, settings = payload["records"], payload["settings"]\n    directory = Path(payload["output"])\n    cache = directory / "generated"\n    todo = [row for row in records if cached_row(cache, row, identity) is None]\n    if payload.get("max_new_images", 0):\n        todo = todo[:payload["max_new_images"]]\n    print(f"ISIC captions: cached={len(records) - sum(cached_row(cache, r, identity) is None for r in records)}, queued={len(todo)}", flush=True)\n    if todo:\n        captioner = factory(payload["model_path"], settings)\n        offset, batch_size = 0, settings["batch_size"]\n        progress = tqdm(total=len(todo), desc="ISIC Florence-Prefix", unit="image")\n        try:\n            while offset < len(todo):\n                group = todo[offset:offset + batch_size]\n                images = []\n                try:\n                    for record in group:\n                        if file_sha256(record["image_path"]) != record["image_sha256"]:\n                            raise ValueError("Image changed after hashing")\n                        with Image.open(record["image_path"]) as source:\n                            images.append(ImageOps.exif_transpose(source).convert("RGB"))\n                    responses = captioner.generate(images)\n                    if len(responses) != len(group):\n                        raise ValueError("Caption batch is incomplete")\n                except torch.cuda.OutOfMemoryError:\n                    if batch_size == 1:\n                        raise RuntimeError("Florence OOM at batch 1; free GPU memory or use a larger GPU. Committed captions are resumable") from None\n                    batch_size = max(1, batch_size // 2)\n                    gc.collect()\n                    torch.cuda.empty_cache()\n                    continue\n                finally:\n                    for image in images:\n                        image.close()\n                for record, (response, limited) in zip(group, responses):\n                    row = make_generation_row(record, response, limited, identity)\n                    atomic_json(cache_path(cache, record["image_id"]), {"identity": identity, "row": row})\n                offset += len(group)\n                progress.update(len(group))\n        finally:\n            progress.close()\n            del captioner\n            gc.collect()\n            torch.cuda.empty_cache()\n    rows = available_rows(cache, records, identity)\n    complete = len(rows) == len(records)\n    path = directory / ("captions.csv" if complete else "captions_partial.csv")\n    atomic_csv(path, rows, COLUMNS)\n    status = {"complete": complete, "rows": len(rows), "expected": len(records), "identity": identity,\n              "csv_path": str(path.resolve()), "csv_sha256": file_sha256(path),\n              "valid_by_split": {role: sum(bool(row["caption_valid"]) for row in rows if row["split"] == role)\n                                 for role in ("train", "val", "test")}}\n    atomic_json(directory / "status.json", status)\n    print(status, flush=True)\n    return status\n\ndef validated_export(payload):\n    directory = Path(payload["output"])\n    status = json.loads((directory / "status.json").read_text(encoding="utf-8"))\n    if not status["complete"]:\n        raise RuntimeError("Caption smoke test/partial output cannot train segmentation; set MAX_NEW_CAPTION_IMAGES=0 and resume")\n    path = directory / "captions.csv"\n    if file_sha256(path) != status["csv_sha256"]:\n        raise ValueError("Caption CSV changed after generation")\n    rows = read_csv_rows(path, COLUMNS)\n    expected = [(r["image_id"], r["split"]) for r in payload["records"]]\n    if [(r["image_id"], r["split"]) for r in rows] != expected:\n        raise ValueError("Wrong/missing/duplicate image IDs or split order in captions.csv")\n    for row in rows:\n        clean, valid, clinical, count = caption_policy(row["source_long_caption"], str(row["generation_token_limit_reached"]).lower() == "true")\n        if row["final_caption"] != clean or row["caption_valid"].lower() != str(valid).lower() or int(row["word_count"]) != count:\n            raise ValueError("Caption policy fields do not match their original Florence response")\n        if row["captioner_fingerprint"] != status["identity"] or row["caption_input_policy"] != POLICY or row["vlm_model"] != MODEL_IDS[METHOD]:\n            raise ValueError("Wrong caption generation provenance")\n    if not any(row["split"] == "train" and row["caption_valid"].lower() == "true" for row in rows):\n        raise ValueError("No usable training captions; inspect CSV and policy, do not force masks/gates on")\n    return rows, status\n\ndef encode_isic_texts(payload):\n    import torch\n    from transformers import AutoTokenizer, CLIPTextModelWithProjection\n    rows, status = validated_export(payload)\n    output = Path(payload["output"])\n    protocol = {"schema": 1, "caption_csv_sha256": status["csv_sha256"],\n                "clip_assets": qwen_asset_identity(payload["clip_path"]), "policy": POLICY,\n                "runtime": runtime_versions(), "code_sha256": payload["code_sha256"]}\n    identity = lock_protocol(output / "text_cache" / "protocol.json", protocol)\n    target = output / "text_cache" / "clip_text.pt"\n    if target.is_file():\n        cached = torch.load(target, map_location="cpu", weights_only=True)\n        validate_text_cache(cached, rows, status["csv_sha256"], identity)\n        atomic_json(output / "text_cache" / "status.json", {"identity": identity, "sha256": file_sha256(target),\n                    "valid": int(cached["valid"].sum()), "total": len(rows), "shape": list(cached["embeddings"].shape)})\n        print("Reusing CLIP text cache:", target, flush=True)\n        return\n    tokenizer = AutoTokenizer.from_pretrained(payload["clip_path"], local_files_only=True)\n    model, info = CLIPTextModelWithProjection.from_pretrained(payload["clip_path"], local_files_only=True,\n                                                            output_loading_info=True)\n    missing = [key for key in info.get("missing_keys", []) if not key.endswith("position_ids")]\n    if missing or info.get("mismatched_keys"):\n        raise ValueError(f"Incomplete CLIP text weights: {missing}")\n    model = model.to("cuda:0").eval().requires_grad_(False)\n    valid = torch.tensor([row["caption_valid"].lower() == "true" for row in rows], dtype=torch.bool)\n    vectors = torch.zeros((len(rows), model.config.projection_dim), dtype=torch.float32)\n    positions = valid.nonzero().flatten().tolist()\n    for start in range(0, len(positions), 128):\n        group = positions[start:start + 128]\n        tokens = tokenizer([rows[i]["final_caption"] for i in group], padding=True, truncation=True,\n                           max_length=model.config.max_position_embeddings, return_tensors="pt")\n        tokens = {key: value.to("cuda:0") for key, value in tokens.items()}\n        with torch.inference_mode():\n            values = torch.nn.functional.normalize(model(**tokens).text_embeds.float(), dim=1)\n        vectors[group] = values.cpu()\n    temporary = target.with_suffix(".pt.tmp")\n    torch.save({"identity": identity, "image_ids": [row["image_id"] for row in rows],\n                "embeddings": vectors, "valid": valid, "caption_csv_sha256": status["csv_sha256"]}, temporary)\n    temporary.replace(target)\n    atomic_json(output / "text_cache" / "status.json", {"identity": identity, "sha256": file_sha256(target),\n                "valid": int(valid.sum()), "total": len(rows), "shape": list(vectors.shape)})\n    print("CLIP vectors:", tuple(vectors.shape), "valid:", int(valid.sum()), flush=True)\n\ndef florence_checkpoint(model_ref, roots):\n    def checked(path):\n        path = Path(path)\n        try:\n            config = json.loads((path / "config.json").read_text(encoding="utf-8"))\n        except (OSError, ValueError):\n            return None\n        if config.get("model_type") != "florence2":\n            return None\n        indexes = list(path.glob("*.index.json"))\n        weights = any((path / name).is_file() for name in ("model.safetensors", "pytorch_model.bin"))\n        for index in indexes:\n            try:\n                shards = set(json.loads(index.read_text(encoding="utf-8")).get("weight_map", {}).values())\n                weights |= bool(shards) and all((path / shard).is_file() for shard in shards)\n            except (OSError, ValueError):\n                continue\n        if not weights:\n            return None\n        backend = "legacy" if "AutoModelForCausalLM" in config.get("auto_map", {}) else "native"\n        if backend == "legacy" and not all((path / name).is_file() for name in (\n                "configuration_florence2.py", "modeling_florence2.py", "processing_florence2.py")):\n            return None\n        return path.resolve(), backend\n\n    requested = Path(model_ref).expanduser()\n    if requested.is_dir():\n        result = checked(requested)\n        if result:\n            return result\n        for config in sorted(requested.rglob("config.json")):\n            result = checked(config.parent)\n            if result:\n                return result\n        raise FileNotFoundError(f"Florence folder is incomplete: {requested}; need config, weights, processor/tokenizer and local model code.")\n    for root in roots:\n        root = Path(root)\n        if not root.is_dir():\n            continue\n        for current, dirs, files in os.walk(root):\n            dirs[:] = sorted(name for name in dirs if name not in {"images", "__pycache__", ".git"})\n            if len(Path(current).relative_to(root).parts) >= 8:\n                dirs[:] = []\n            if "config.json" in files:\n                result = checked(current)\n                if result:\n                    return result\n    raise FileNotFoundError(\n        f"No complete local Florence-2 checkpoint found for {model_ref}. "\n        "Attach notebook-mt-moi/Florence-2-base or set FLORENCE_MODEL_REF to the correct local folder. No download is attempted."\n    )\n\ndef offline_caption_wheels(roots, destination):\n    import zipfile\n    destination = Path(destination)\n    wheel_dirs = set()\n    prefixes = ("transformers-", "tokenizers-", "huggingface_hub-", "accelerate-", "einops-", "timm-")\n    for root in roots:\n        root = Path(root)\n        if not root.is_dir():\n            continue\n        for current, dirs, files in os.walk(root):\n            dirs[:] = sorted(name for name in dirs if name not in {"images", "__pycache__", ".git", "Florence-2-base"})\n            if len(Path(current).relative_to(root).parts) >= 7:\n                dirs[:] = []\n            for name in sorted(files):\n                path = Path(current) / name\n                if name.endswith(".whl"):\n                    wheel_dirs.add(str(path.parent))\n                elif name.lower() in {"offline_wheels.zip", "wheels.zip"}:\n                    with zipfile.ZipFile(path) as archive:\n                        for member in archive.infolist():\n                            filename = Path(member.filename.replace("\\\\", "/")).name\n                            if not filename.endswith(".whl") or not filename.startswith(prefixes):\n                                continue\n                            destination.mkdir(parents=True, exist_ok=True)\n                            target = destination / filename\n                            if not target.is_file() or target.stat().st_size != member.file_size:\n                                with archive.open(member) as source, target.open("wb") as handle:\n                                    shutil.copyfileobj(source, handle)\n                            wheel_dirs.add(str(destination))\n    return sorted(wheel_dirs)\n\ndef caption_runtime_environment(backend, asset_root, runtime_dir):\n    import subprocess\n    import sys\n    runtime_dir = Path(runtime_dir)\n    runtime_dir.mkdir(parents=True, exist_ok=True)\n    env = dict(os.environ)\n    env.update(TOKENIZERS_PARALLELISM="false", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",\n               HF_MODULES_CACHE=str(runtime_dir / "hf_modules"), PYTHONUNBUFFERED="1")\n    target = runtime_dir / "legacy_packages"\n    probe = (\n        "import transformers; "\n        + ("assert transformers.__version__.startswith(\'4.49.\'), transformers.__version__; " if backend == "legacy" else "")\n        + "from transformers import AutoProcessor, AutoTokenizer, CLIPModel; "\n        + ("from transformers import AutoModelForCausalLM; " if backend == "legacy" else\n           "from transformers import Florence2ForConditionalGeneration; ")\n        + "import einops, timm, accelerate; print(transformers.__version__)"\n    )\n    if target.is_dir():\n        env["PYTHONPATH"] = str(target) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")\n    result = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)\n    if result.returncode == 0:\n        print("Caption subprocess Transformers:", result.stdout.strip())\n        return env\n    if backend != "legacy":\n        raise RuntimeError("The local native Florence checkpoint cannot load in this runtime. "\n                           "Use a matching native Transformers installation or the attached legacy model + offline 4.49 wheels.\\n"\n                           + result.stderr[-3000:])\n    wheel_dirs = offline_caption_wheels([Path(asset_root), Path("/kaggle/input")], runtime_dir / "wheels")\n    if not wheel_dirs:\n        raise FileNotFoundError(\n            "The attached Microsoft Florence checkpoint needs Transformers 4.49.x. "\n            f"No offline wheels found under {asset_root}. Attach offline_wheels.zip/wheels.zip or a wheel folder "\n            "with transformers==4.49.0, tokenizers 0.21.x and huggingface-hub <1 compatible with this Python version. "\n            "The notebook never enables Internet or modifies the kernel\'s Transformers."\n        )\n    command = [sys.executable, "-m", "pip", "install", "--no-index", "--only-binary=:all:",\n               "--no-deps", "--upgrade", "--target", str(target), "transformers==4.49.0",\n               "tokenizers>=0.21,<0.22", "huggingface-hub>=0.26,<1"]\n    import importlib.util\n    for module in ("einops", "timm", "accelerate"):\n        if importlib.util.find_spec(module) is None:\n            command.append(module)\n    for directory in wheel_dirs:\n        command.extend(["--find-links", directory])\n    print("Preparing isolated Florence dependencies from attached offline wheels; the notebook kernel is unchanged.")\n    installed = subprocess.run(command, env=env, capture_output=True, text=True)\n    if installed.returncode:\n        raise RuntimeError("Offline Florence dependency setup failed. Wheels must match Kaggle\'s Python/platform.\\n"\n                           + installed.stdout[-2000:] + installed.stderr[-3000:])\n    base_pythonpath = os.environ.get("PYTHONPATH", "")\n    env["PYTHONPATH"] = str(target) + (os.pathsep + base_pythonpath if base_pythonpath else "")\n    checked = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)\n    if checked.returncode:\n        raise RuntimeError("The offline Florence environment still lacks a compatible dependency. "\n                           "No kernel packages were changed.\\n" + checked.stderr[-4000:])\n    print("Isolated caption subprocess Transformers:", checked.stdout.strip())\n    return env\n\ndef florence_worker_main(request_path, phase, *, generate_records, encode_texts):\n    import torch\n\n    payload = json.loads(Path(request_path).read_text(encoding="utf-8"))\n    validate_caption_records(payload["records"])\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a CUDA GPU")\n    settings = payload["settings"]\n    if type(payload.get("max_new_images", 0)) is not int or payload.get("max_new_images", 0) < 0:\n        raise ValueError("max_new_images must be a nonnegative integer")\n    for key in ("batch_size", "max_new_tokens"):\n        if type(settings[key]) is not int or settings[key] < 1:\n            raise ValueError(f"{key} must be a positive integer")\n    if type(settings["seed"]) is not int or not 0 <= settings["seed"] < 2**32:\n        raise ValueError("Caption seed must be an integer in [0, 2**32)")\n    torch.manual_seed(settings["seed"])\n    torch.cuda.manual_seed_all(settings["seed"])\n    if phase == "generate":\n        from importlib.metadata import version\n        if not version("transformers").startswith("4.49."):\n            raise RuntimeError("Legacy Florence generation must use the isolated Transformers 4.49 runtime")\n        path, count = validate_florence_prefix_folder(payload["model_path"])\n        protocol = {"schema": 1, "method": METHOD, "model_id": MODEL_ID,\n                    "model_assets": checkpoint_asset_identity(path), "parameters": count,\n                    "prompt": TASK_PROMPT, "decoding": "greedy_beam1_prefix_cleanup_first8",\n                    "dtype": "float32",\n                    "policy": "isic_visible_attributes_v1",\n                    "images": [{key: row[key] for key in ("image_id", "split", "image_sha256")}\n                               for row in payload["records"]],\n                    "settings": settings, "runtime": runtime_versions(), "code_sha256": payload["code_sha256"]}\n        output = Path(payload["output"])\n        identity = lock_protocol(output / "generation_protocol" / "protocol.json", protocol)\n        if not (output / "status.json").exists() and (output / "captions.csv").exists():\n            raise ValueError("Unregistered captions.csv; use a fresh output root")\n        generate_records(payload, identity)\n    elif phase == "encode":\n        encode_texts(payload)\n    else:\n        raise ValueError("Phase must be generate or encode")\n\n\nif __name__ == "__main__":\n    import torch\n    torch.set_float32_matmul_precision("highest")\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    florence_worker_main(sys.argv[1], sys.argv[2], generate_records=generate_isic_records, encode_texts=encode_isic_texts)\n', 'isic_caption_web.py': '"""ISIC Florence-Prefix generation/resume and a portable fresh-image adapter.\n\nisic_caption_core.py is embedded by the notebook builder from the existing,\ntested ISIC caption pipeline. Transformers never imports in the training kernel.\n"""\nfrom __future__ import annotations\n\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport tempfile\n\nimport torch\n\nimport isic_caption_core as core\nimport tacs_visual_web as io\n\n\ndef validate_clip_assets(folder):\n    folder = Path(folder)\n    config = io.read_json(folder / "config.json")\n    if (config.get("model_type") != "clip" or config.get("projection_dim") != 512\n            or config.get("vision_config", {}).get("patch_size") != 32):\n        raise ValueError("Use a full local CLIP ViT-B/32, not a different text embedding space")\n    if not (folder / "tokenizer_config.json").is_file() or not (\n            (folder / "tokenizer.json").is_file() or all((folder / name).is_file() for name in ("vocab.json", "merges.txt"))):\n        raise FileNotFoundError("CLIP tokenizer files are incomplete; fix Inputs before generating captions")\n    weights = list(folder.glob("*.safetensors")) + list(folder.glob("pytorch_model*.bin"))\n    if not weights or any(path.stat().st_size == 0 for path in weights):\n        raise FileNotFoundError("CLIP weights are missing/empty")\n    for index in folder.glob("*.index.json"):\n        names = set(io.read_json(index).get("weight_map", {}).values())\n        if not names or any(not io.bundle_path(folder, name).is_file() for name in names):\n            raise FileNotFoundError("CLIP weight index has missing shards")\n    return folder\n\n\ndef prepare_captions(frames, florence_ref, clip_ref, offline_assets, output, settings, runtime_dir):\n    if settings.get("precision") != "float32":\n        raise ValueError("This deployment protocol fixes Florence/CLIP precision to float32")\n    output = Path(output).resolve()\n    output.mkdir(parents=True, exist_ok=True)\n    roots = [Path(offline_assets), Path("/kaggle/input")]\n    florence, backend = core.florence_checkpoint(florence_ref, roots)\n    if backend != "legacy":\n        raise ValueError("This method fixes Microsoft Florence-2-base legacy + isolated Transformers 4.49")\n    core.validate_florence_prefix_folder(florence)\n    clip = core.resolve_clip_model(clip_ref, roots)\n    validate_clip_assets(clip)\n    records = [{key: row[key] for key in ("image_id", "image_path", "split", "image_sha256")}\n               for role in ("train", "val", "test") for row in frames[role]]\n    # Explicit whitelist prevents masks, diagnoses and clinical metadata entering\n    # either the caption prompt or its generation worker payload.\n    core.validate_caption_records(records)\n    worker = Path(runtime_dir) / "isic_caption_core.py"\n    payload = {"records": records, "output": str(output), "settings": dict(settings),\n        "model_path": str(florence), "clip_path": str(clip), "max_new_images": 0,\n        "code_sha256": io.file_sha256(worker)}\n    request = output / "request.json"\n    io.write_json(request, payload)\n    env = core.caption_runtime_environment("legacy", offline_assets, output / "caption_runtime")\n    for phase in ("generate", "encode"):\n        core.run_worker(worker, request, phase, env, output / f"{phase}.log")\n    rows, status = core.validated_export(payload)\n    cache_file = output / "text_cache/clip_text.pt"\n    cache = torch.load(cache_file, map_location="cpu", weights_only=True)\n    core.validate_text_cache(cache, rows, status["csv_sha256"], cache["identity"])\n    if cache["embeddings"].shape[1] != 512:\n        raise ValueError("Unexpected CLIP text dimension")\n    valid_by_split = {role: sum(row["split"] == role and row["caption_valid"].lower() == "true" for row in rows)\n                      for role in ("train", "val", "test")}\n    if valid_by_split["train"] == 0:\n        raise ValueError("No valid training captions; inspect the CSV before calling this full_both")\n    io.write_json(output / "web_caption_settings.json", {"settings": settings, "policy": core.POLICY,\n        "method": core.METHOD, "model_id": core.MODEL_ID, "valid_by_split": valid_by_split,\n        "florence_assets": core.checkpoint_asset_identity(florence), "clip_assets": core.checkpoint_asset_identity(clip),\n        "source": "pixel_only_automatic_caption_not_ground_truth"})\n    return cache_file, {"florence": str(florence), "clip": str(clip), "offline_assets_root": str(offline_assets)}\n\n\nclass _LocalCaptionEngine:\n    def __init__(self, root, device):\n        from transformers import AutoTokenizer, CLIPTextModelWithProjection\n        self.root = Path(root)\n        registered = io.read_json(self.root / "caption_settings.json")\n        if registered["policy"] != core.POLICY or registered["method"] != core.METHOD:\n            raise ValueError("Caption policy/method differs from the training export")\n        if registered["settings"].get("precision") != "float32":\n            raise ValueError("Caption precision differs from the FP32 training export")\n        if not importlib.metadata.version("transformers").startswith("4.49."):\n            raise RuntimeError("The local caption worker requires Transformers 4.49.x")\n        settings = {**registered["settings"], "device": str(device)}\n        self.captioner = core.IsicFlorencePrefixCaptioner(self.root / "caption_models/florence", settings)\n        self.device = torch.device(device)\n        self.tokenizer = AutoTokenizer.from_pretrained(str(self.root / "caption_models/clip"), local_files_only=True)\n        self.clip, info = CLIPTextModelWithProjection.from_pretrained(str(self.root / "caption_models/clip"),\n            local_files_only=True, output_loading_info=True)\n        missing = [key for key in info.get("missing_keys", []) if not key.endswith("position_ids")]\n        if missing or info.get("mismatched_keys"):\n            raise ValueError("The exported CLIP text tower did not load completely")\n        self.clip = self.clip.to(self.device).eval().requires_grad_(False)\n\n    def caption(self, image):\n        raw, limited = self.captioner.generate([io.read_rgb(image)])[0]\n        final, valid, clinical, words = core.caption_policy(raw, limited)\n        vector = torch.zeros(512, dtype=torch.float32)\n        if valid:\n            tokens = self.tokenizer([final], padding=True, truncation=True,\n                max_length=self.clip.config.max_position_embeddings, return_tensors="pt")\n            with torch.inference_mode():\n                values = self.clip(**{key: value.to(self.device) for key, value in tokens.items()}).text_embeds\n                values = torch.nn.functional.normalize(values.float(), dim=1)[0].cpu()\n            # Copy into a normal tensor: no in-place mutation of an inference\n            # tensor outside InferenceMode (the previous web caption failure).\n            vector = values.clone()\n        row = {"raw_caption": raw, "final_caption": final, "caption_valid": valid,\n            "clinical_term_detected": clinical, "word_count": words, "caption_policy": core.POLICY}\n        return row, vector, torch.tensor(valid, dtype=torch.bool)\n\n\nclass BundledCaptionEngine:\n    """Persistent local models on 4.49; isolated offline child on another version.\n\nFor a production web deployment, keep this object in a dedicated compatible\nworker so the local models are reused rather than loaded for every request.\n"""\n    def __init__(self, bundle, device="cpu", offline_assets=None, runtime=None):\n        self.root, self.device = Path(bundle).resolve(), str(device)\n        self.local = None\n        try:\n            version = importlib.metadata.version("transformers")\n        except importlib.metadata.PackageNotFoundError:\n            version = ""\n        if version.startswith("4.49."):\n            self.local = _LocalCaptionEngine(self.root, device)\n        else:\n            assets = offline_assets or os.getenv("ISIC_CAPTION_OFFLINE_ASSETS") or self.root / "offline_wheels"\n            runtime = runtime or Path(tempfile.gettempdir()) / "isic_caption_runtime" / io.file_sha256(self.root / "caption_settings.json")[:16]\n            self.environment = core.caption_runtime_environment("legacy", assets, runtime)\n\n    def caption(self, image):\n        if self.local is not None:\n            return self.local.caption(image)\n        with tempfile.TemporaryDirectory(prefix="isic_caption_") as directory:\n            directory = Path(directory)\n            image_file, result_file = directory / "query.png", directory / "result.pt"\n            io.read_rgb(image).save(image_file)\n            command = [sys.executable, str(self.root / "runtime/isic_caption_web.py"), "--bundle", str(self.root),\n                "--image", str(image_file), "--result", str(result_file), "--device", self.device]\n            completed = subprocess.run(command, env=self.environment, text=True, capture_output=True, timeout=900)\n            if completed.returncode:\n                raise RuntimeError("Bundled caption worker failed: " + completed.stderr[-4000:])\n            result = torch.load(result_file, map_location="cpu", weights_only=True)\n            return result["row"], result["vector"], result["valid"]\n\n\ndef probe_export(bundle, device, offline_assets, runtime):\n    # Force generation even for a known image: the segmentor\'s pixel cache must\n    # not hide missing Florence/CLIP weights in a supposedly complete export.\n    engine = BundledCaptionEngine(bundle, device, offline_assets, runtime)\n    row, vector, valid = engine.caption(Path(bundle) / "example/query.png")\n    if vector.shape != (512,) or not torch.isfinite(vector).all() or (not bool(valid) and vector.count_nonzero()):\n        raise ValueError("Fresh exported caption engine has invalid vector/mask output")\n    return {"fresh_caption": row, "valid": bool(valid), "embedding_shape": list(vector.shape)}\n\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser()\n    for name in ("bundle", "image", "result"):\n        parser.add_argument("--" + name, required=True)\n    parser.add_argument("--device", default="cpu")\n    arguments = parser.parse_args()\n    torch.set_num_threads(2)\n    engine = _LocalCaptionEngine(arguments.bundle, arguments.device)\n    row, vector, valid = engine.caption(io.read_rgb(arguments.image))\n    torch.save({"row": row, "vector": vector, "valid": valid}, arguments.result)\n'}
for name, source in EXPORTERS.items():
    target = EXPORT_CODE_DIR / name
    if target.is_file() and target.read_text(encoding="utf-8") != source:
        raise ValueError("Exporter changed under this output. Choose a new OUTPUT_DIR.")
    target.write_text(source, encoding="utf-8")
    loaded = sys.modules.get(Path(name).stem)
    if loaded and Path(loaded.__file__).resolve() != target.resolve():
        raise RuntimeError("Another exporter is imported. Restart the kernel and Run All.")
sys.path.insert(0, str(EXPORT_CODE_DIR.resolve()))
import isic_export_only as shared_export
import isic_mtacs_export_only as exporter


## 2. Khớp checkpoint, split, pool và captions đã train

Runtime của chính output cũ được ưu tiên và phải khớp `run_binding.json`. Chỉ dùng fallback nhúng khi đúng hash;
không đoán kiến trúc hoặc chia lại dữ liệu. Kiểm SHA ảnh/mask/model/CSV/cache, policy và mask text.
Nếu thiếu file caption cũ, giữ output làm Dataset đầy đủ; không sinh CSV mới để ghép vào checkpoint này.


In [ ]:
CONTEXT = exporter.prepare(
    SOURCE_RUN_DIR, ISIC_ROOT, DINOV3_REPO_DIR, DINOV3_WEIGHTS, OUTPUT_DIR, FALLBACK_SOURCES,
    caption_output=CAPTION_OUTPUT_DIR, florence_ref=FLORENCE_MODEL_REF, clip_ref=CLIP_MODEL_REF,
    offline_assets=OFFLINE_ASSETS_ROOT, device=DEVICE, feature_batch_size=FEATURE_BATCH_SIZE,
    run_full_test=RUN_FULL_TEST,
)
display(CONTEXT["snapshot"])
display(CONTEXT["caption_coverage"])
print("Source run:", CONTEXT["run"])


## 3. Xuất model và dữ liệu portable

Giữ nguyên best.pt gốc; tách phần học được thành `selector.pt` và `segmentor.pt`, kèm frozen backbone.
Copy đúng Florence/CLIP đã đăng ký, dùng nguyên CSV/embedding. Giải phóng model trước khi kiểm caption và nạp lại ở process khác.
Không copy last.pt/optimizer, per-image generated caches, legacy_packages hay log lớn vào ZIP.


In [ ]:
CONTEXT = shared_export.build(CONTEXT)
print("Staged bundle:", CONTEXT["stage"])
display(shared_export.read_json(CONTEXT["stage"] / "metrics.json"))
for name in ("models/selector.pt", "models/segmentor.pt", "candidate_bank.pt", "captions.csv", "text_embeddings.pt"):
    path = CONTEXT["stage"] / name
    print(name, round(path.stat().st_size / 1024**2, 2), "MiB")


## 4. Kiểm nạp lại và ảnh mới

Process caption độc lập nạp Florence/CLIP từ bundle, bắt buộc sinh một caption mới từ pixels và kiểm CLIP tái tạo
vector của một caption đã train. Process segmentation độc lập kiểm logits với checkpoint, context ID, hai map attention,
và forward full_both khi dùng caption vừa sinh. Không thay CSV đã train. Chỉ thành công mới ghi `ready`.
Một ảnh minh họa không phải metric toàn test; mặc định `metrics.json` để test=null nếu chưa chạy đánh giá test.


In [ ]:
BUNDLE_DIR = exporter.verify_and_publish(CONTEXT, device=COLD_RELOAD_DEVICE, caption_device=CAPTION_VERIFY_DEVICE)
display(shared_export.read_json(BUNDLE_DIR / "smoke_test.json"))
display(shared_export.read_json(BUNDLE_DIR / "example/fresh_caption_check.json"))
print("Verified bundle:", BUNDLE_DIR)


In [ ]:
from IPython.display import display, Image as DisplayImage
prediction = shared_export.read_json(BUNDLE_DIR / "example/prediction.json")
display({"query_caption": prediction["evidence"]["query_caption"],
         "candidate_id": prediction["evidence"]["candidate"]["image_id"],
         "candidate_caption": prediction["evidence"]["candidate"]["caption"],
         "fusion": prediction["evidence"]["fusion"], "not_test_set_metrics": True})
for name in ("mask", "overlay", "query_segmentor", "candidate_selector"):
    print(name)
    display(DisplayImage(filename=str(BUNDLE_DIR / "example" / (name + ".png")), width=420))


## 5. Tải từng phần ZIP

Tải **tất cả** `.zip.partNNN` và hai file nhỏ `download_manifest.json`, `join_downloads.py` vào cùng folder máy tính.
Chạy `python join_downloads.py`, rồi giải nén ZIP đã ghép. Script kiểm SHA256 và không cần PyTorch.
Các phần không phải ZIP độc lập. Nếu web dùng Python/platform khác Kaggle, wheels native phải phù hợp platform;
model weights/CSV/runtime vẫn giữ nguyên. Máy web cần môi trường Python/PyTorch tương thích.


In [ ]:
DOWNLOAD_DIR = exporter.package_downloads(CONTEXT, part_mib=DOWNLOAD_PART_MIB)
from IPython.display import FileLink
download = shared_export.read_json(DOWNLOAD_DIR / "download_manifest.json")
print("Download", len(download["parts"]), "parts + download_manifest.json + join_downloads.py")
for item in download["parts"]:
    print(item["name"], round(item["bytes"] / 1024**2, 2), "MiB")
    display(FileLink(str(DOWNLOAD_DIR / item["name"])))
for name in ("download_manifest.json", "join_downloads.py"):
    display(FileLink(str(DOWNLOAD_DIR / name)))
